In [3]:
# Phase 1: Environment setup and project folder structure
# This cell is self-contained and imports everything it uses.

import os
import sys
import json
import csv
import hashlib
import logging
import time
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

import requests



In [4]:
# ---------------------------------------------------------------------------
# Project root and folder structure
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"

FOLDERS = [
    PROJECT_ROOT / "data" / "raw" / "ncdc_reports",
    PROJECT_ROOT / "data" / "interim",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "logs",
]

for folder in FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)


In [5]:
# ---------------------------------------------------------------------------
# Logging setup
# ---------------------------------------------------------------------------

LOG_FILE = PROJECT_ROOT / "logs" / "pipeline.log"

logger = logging.getLogger("lassa_fever_nimr")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setFormatter(
    logging.Formatter("%(levelname)s | %(message)s")
)
logger.addHandler(console_handler)

In [6]:
# ---------------------------------------------------------------------------
# Confirm setup
# ---------------------------------------------------------------------------

logger.info("Project root: %s", PROJECT_ROOT.resolve())
for folder in FOLDERS:
    logger.info("Confirmed folder exists: %s", folder.resolve())

logger.info("Phase 1 complete. Environment and folder structure ready.")

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("Log file:", LOG_FILE.resolve())

INFO | Project root: C:\Users\USER\lassa_fever_nimr
INFO | Confirmed folder exists: C:\Users\USER\lassa_fever_nimr\data\raw\ncdc_reports
INFO | Confirmed folder exists: C:\Users\USER\lassa_fever_nimr\data\interim
INFO | Confirmed folder exists: C:\Users\USER\lassa_fever_nimr\data\processed
INFO | Confirmed folder exists: C:\Users\USER\lassa_fever_nimr\outputs
INFO | Confirmed folder exists: C:\Users\USER\lassa_fever_nimr\logs
INFO | Phase 1 complete. Environment and folder structure ready.
PROJECT_ROOT: C:\Users\USER\lassa_fever_nimr
Log file: C:\Users\USER\lassa_fever_nimr\logs\pipeline.log


In [7]:
# Phase 2: NCDC report catalogue
# This cell is self-contained and imports everything it uses.

import re
import time
import csv
import logging
from pathlib import Path
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup


In [8]:
# ---------------------------------------------------------------------------
# Reuse project structure defined in Phase 1 (recreate paths defensively
# in case this cell is run in a fresh kernel)
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

NCDC_URL = (
    "https://www.ncdc.gov.ng/diseases/sitreps/?cat=5"
    "&name=An+update+of+Lassa+fever+outbreak+in+Nigeria"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

REQUEST_TIMEOUT = 30
MAX_RETRIES = 4
RETRY_DELAY_SECONDS = 5


In [9]:
# ---------------------------------------------------------------------------
# Polite fetch with retries
# ---------------------------------------------------------------------------

def fetch_with_retries(url, headers, timeout, max_retries, retry_delay):
    last_exception = None
    for attempt in range(1, max_retries + 1):
        try:
            logger.info("Fetch attempt %d for %s", attempt, url)
            response = requests.get(url, headers=headers, timeout=timeout)
            logger.info(
                "Attempt %d returned status %d (final URL: %s)",
                attempt, response.status_code, response.url
            )
            if response.status_code == 200:
                return response
            if response.status_code == 500:
                logger.warning("Received HTTP 500, will retry after delay.")
                time.sleep(retry_delay)
                continue
            logger.warning("Unexpected status %d, will retry after delay.", response.status_code)
            time.sleep(retry_delay)
        except requests.RequestException as exc:
            last_exception = exc
            logger.warning("Request exception on attempt %d: %s", attempt, exc)
            time.sleep(retry_delay)
    if last_exception is not None:
        raise last_exception
    return None

response = fetch_with_retries(
    NCDC_URL, HEADERS, REQUEST_TIMEOUT, MAX_RETRIES, RETRY_DELAY_SECONDS
)

if response is None or response.status_code != 200:
    status = response.status_code if response is not None else "no response"
    logger.error("Failed to retrieve NCDC page after retries. Final status: %s", status)
    raise RuntimeError(
        f"Could not retrieve NCDC sitrep page after {MAX_RETRIES} attempts. "
        f"Final status: {status}. Stopping rather than continuing to hammer the site."
    )

retrieved_at = datetime.now(timezone.utc).isoformat()
logger.info("Successfully retrieved page. Status: %d. Final URL: %s", response.status_code, response.url)


INFO | Fetch attempt 1 for https://www.ncdc.gov.ng/diseases/sitreps/?cat=5&name=An+update+of+Lassa+fever+outbreak+in+Nigeria
INFO | Attempt 1 returned status 200 (final URL: https://www.ncdc.gov.ng/diseases/sitreps/?cat=5&name=An+update+of+Lassa+fever+outbreak+in+Nigeria)
INFO | Successfully retrieved page. Status: 200. Final URL: https://www.ncdc.gov.ng/diseases/sitreps/?cat=5&name=An+update+of+Lassa+fever+outbreak+in+Nigeria


In [10]:
# ---------------------------------------------------------------------------
# Save raw HTML for manual inspection
# ---------------------------------------------------------------------------

raw_html_path = INTERIM_DIR / "ncdc_sitrep_page_raw.html"
raw_html_path.write_text(response.text, encoding="utf-8")
logger.info("Saved raw HTML to %s (%d bytes)", raw_html_path, len(response.text))


INFO | Saved raw HTML to C:\Users\USER\lassa_fever_nimr\data\interim\ncdc_sitrep_page_raw.html (411392 bytes)


In [11]:
# ---------------------------------------------------------------------------
# Parse page and attempt candidate extraction strategies
# ---------------------------------------------------------------------------

soup = BeautifulSoup(response.text, "html.parser")

# Strategy A: standard anchor tags linking to PDFs
pdf_links = []
for a in soup.find_all("a", href=True):
    href = a["href"].strip()
    text = a.get_text(strip=True)
    if href.lower().endswith(".pdf"):
        pdf_links.append({"href": href, "text": text})

# Strategy B: anchor tags whose visible text looks like a sitrep title
sitrep_keyword_links = []
keyword_pattern = re.compile(r"lassa|sitrep|situation report|epidemiolog", re.IGNORECASE)
for a in soup.find_all("a", href=True):
    href = a["href"].strip()
    text = a.get_text(strip=True)
    if text and keyword_pattern.search(text):
        sitrep_keyword_links.append({"href": href, "text": text})

# Strategy C: table rows, in case reports are listed in a table rather than a link list
table_row_count = len(soup.find_all("tr"))

# Strategy D: any element with a data attribute or script tag suggesting JS-driven loading
script_tags = soup.find_all("script")
possible_js_driven = any(
    "sitrep" in (tag.string or "").lower() if tag.string else False
    for tag in script_tags
)

print("Status code:", response.status_code)
print("Final URL:", response.url)
print("Raw HTML saved to:", raw_html_path.resolve())
print()
print("Strategy A: direct PDF links found:", len(pdf_links))
print("Strategy B: keyword-matched anchor links found:", len(sitrep_keyword_links))
print("Strategy C: table row count on page:", table_row_count)
print("Strategy D: possible JS-driven content detected:", possible_js_driven)
print()
print("Sample of Strategy A results (up to 5):")
for item in pdf_links[:5]:
    print(" -", item)
print()
print("Sample of Strategy B results (up to 5):")
for item in sitrep_keyword_links[:5]:
    print(" -", item)

Status code: 200
Final URL: https://www.ncdc.gov.ng/diseases/sitreps/?cat=5&name=An+update+of+Lassa+fever+outbreak+in+Nigeria
Raw HTML saved to: C:\Users\USER\lassa_fever_nimr\data\interim\ncdc_sitrep_page_raw.html

Strategy A: direct PDF links found: 484
Strategy B: keyword-matched anchor links found: 2
Strategy C: table row count on page: 490
Strategy D: possible JS-driven content detected: False

Sample of Strategy A results (up to 5):
 - {'href': '/themes/common/files/sitreps/2c311ea5d133f34c14cf42ee191c99d6.pdf', 'text': ''}
 - {'href': '/themes/common/files/sitreps/c657ffd9e1497fd496235d98f0bc029a.pdf', 'text': ''}
 - {'href': '/themes/common/files/sitreps/b0fedda076a0b27d21d5a09678dd69a0.pdf', 'text': ''}
 - {'href': '/themes/common/files/sitreps/fe81caea94c6909fe650cd814949a985.pdf', 'text': ''}
 - {'href': '/themes/common/files/sitreps/96c09a30808c7b5d5f3db045fe340770.pdf', 'text': ''}

Sample of Strategy B results (up to 5):
 - {'href': '/reports/weekly', 'text': 'Weekly Epid

In [12]:
# Phase 2 (continued): inspect table row structure around PDF links
# This cell is self-contained and imports everything it uses.

from pathlib import Path
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
raw_html_path = INTERIM_DIR / "ncdc_sitrep_page_raw.html"

html_text = raw_html_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html_text, "html.parser")

# Find all <tr> elements that contain a link to a PDF
rows_with_pdf = []
for tr in soup.find_all("tr"):
    a_tag = tr.find("a", href=True)
    if a_tag and a_tag["href"].strip().lower().endswith(".pdf"):
        rows_with_pdf.append(tr)

print("Total <tr> rows containing a PDF link:", len(rows_with_pdf))
print()

# Print the cell structure (tag name and text) for the first 3 matching rows
for i, tr in enumerate(rows_with_pdf[:3], start=1):
    print(f"--- Row {i} ---")
    cells = tr.find_all(["td", "th"])
    print("Number of cells in row:", len(cells))
    for j, cell in enumerate(cells):
        text = cell.get_text(strip=True)
        print(f"  Cell {j}: '{text}'")
    print("Raw row HTML:")
    print(tr.prettify()[:1500])
    print()

Total <tr> rows containing a PDF link: 484

--- Row 1 ---
Number of cells in row: 3
  Cell 0: '1'
  Cell 1: 'An update of Lassa fever outbreak in Nigeria for Week 36'
  Cell 2: ''
Raw row HTML:
<tr>
 <td>
  1
 </td>
 <td>
  An update of Lassa fever outbreak in Nigeria for Week 36
 </td>
 <!--<td>05 September 2026</td>-->
 <td>
  <a class="black-text" download="An update of Lassa fever outbreak in Nigeria_050926_36.pdf" href="/themes/common/files/sitreps/2c311ea5d133f34c14cf42ee191c99d6.pdf">
   <i class="fa fa-download">
   </i>
  </a>
 </td>
</tr>


--- Row 2 ---
Number of cells in row: 3
  Cell 0: '2'
  Cell 1: 'An update of Lassa fever outbreak in Nigeria for Week 35'
  Cell 2: ''
Raw row HTML:
<tr>
 <td>
  2
 </td>
 <td>
  An update of Lassa fever outbreak in Nigeria for Week 35
 </td>
 <!--<td>29 August 2026</td>-->
 <td>
  <a class="black-text" download="An update of Lassa fever outbreak in Nigeria_290826_35.pdf" href="/themes/common/files/sitreps/c657ffd9e1497fd496235d98f0bc029a

In [13]:
# Phase 2 (final): build and save the NCDC report catalogue
# This cell is self-contained and imports everything it uses.

import re
import csv
import hashlib
import logging
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin

from bs4 import BeautifulSoup, Comment

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

BASE_URL = "https://www.ncdc.gov.ng"
raw_html_path = INTERIM_DIR / "ncdc_sitrep_page_raw.html"

if not raw_html_path.exists():
    raise RuntimeError(
        f"Raw HTML file not found at {raw_html_path}. "
        "Re-run the earlier fetch cell before running this one."
    )

html_text = raw_html_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html_text, "html.parser")

WEEK_PATTERN = re.compile(r"week\s+(\d+)", re.IGNORECASE)
COMMENT_DATE_PATTERN = re.compile(r"<td>\s*(.*?)\s*</td>", re.IGNORECASE)
FILENAME_DATE_WEEK_PATTERN = re.compile(r"_(\d{6})_(\d+)\.pdf$", re.IGNORECASE)

def parse_comment_date(tr_tag):
    """Search a table row's HTML comments for a date like '22 August 2026'."""
    for node in tr_tag.find_all(string=lambda t: isinstance(t, Comment)):
        match = COMMENT_DATE_PATTERN.search(str(node))
        if match:
            raw_text = match.group(1).strip()
            for fmt in ("%d %B %Y", "%d %b %Y", "%B %d, %Y"):
                try:
                    return raw_text, datetime.strptime(raw_text, fmt).date().isoformat()
                except ValueError:
                    continue
            return raw_text, None
    return None, None

def parse_filename_date_week(download_attr):
    """Parse the DDMMYY_WW pattern embedded in the PDF download filename."""
    if not download_attr:
        return None, None
    match = FILENAME_DATE_WEEK_PATTERN.search(download_attr)
    if not match:
        return None, None
    ddmmyy, week = match.group(1), match.group(2)
    try:
        parsed_date = datetime.strptime(ddmmyy, "%d%m%y").date().isoformat()
    except ValueError:
        parsed_date = None
    return parsed_date, week

rows_with_pdf = []
for tr in soup.find_all("tr"):
    a_tag = tr.find("a", href=True)
    if a_tag and a_tag["href"].strip().lower().endswith(".pdf"):
        rows_with_pdf.append((tr, a_tag))

catalogue_rows = []
retrieved_at = datetime.now(timezone.utc).isoformat()

parse_failures = []

for tr, a_tag in rows_with_pdf:
    cells = tr.find_all(["td", "th"])
    row_sequence_number = cells[0].get_text(strip=True) if len(cells) > 0 else ""
    report_title = cells[1].get_text(strip=True) if len(cells) > 1 else ""

    href = a_tag["href"].strip()
    source_url = urljoin(BASE_URL, href)
    download_filename = a_tag.get("download", "").strip()

    week_match = WEEK_PATTERN.search(report_title)
    sitrep_week = week_match.group(1) if week_match else None

    comment_date_raw, comment_date_iso = parse_comment_date(tr)
    filename_date_iso, filename_week = parse_filename_date_week(download_filename)

    # Prefer the HTML comment date; fall back to the filename-derived date.
    best_date_iso = comment_date_iso or filename_date_iso

    report_id = hashlib.sha256(source_url.encode("utf-8")).hexdigest()[:16]

    if not report_title or not source_url:
        parse_failures.append({"row_sequence_number": row_sequence_number, "href": href})
        continue

    catalogue_rows.append({
        "report_id": report_id,
        "report_title": report_title,
        "source_url": source_url,
        "catalogue_retrieved_at_utc": retrieved_at,
        "row_sequence_number": row_sequence_number,
        "sitrep_week": sitrep_week,
        "report_date_iso": best_date_iso,
        "report_date_source": "html_comment" if comment_date_iso else (
            "filename" if filename_date_iso else "unresolved"
        ),
        "download_filename": download_filename,
    })

logger.info("Parsed %d catalogue rows from %d candidate table rows.",
            len(catalogue_rows), len(rows_with_pdf))
if parse_failures:
    logger.warning("Skipped %d rows due to missing title or URL.", len(parse_failures))

if len(catalogue_rows) == 0:
    raise RuntimeError(
        "Catalogue is empty after parsing. Stopping before the download stage, "
        "as required. Inspect the raw HTML and parsing logic before proceeding."
    )

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
fieldnames = list(catalogue_rows[0].keys())
with catalogue_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(catalogue_rows)

logger.info("Catalogue saved to %s", catalogue_csv_path)

print("Catalogue rows created:", len(catalogue_rows))
print("Rows skipped (missing title or URL):", len(parse_failures))
print("Catalogue saved to:", catalogue_csv_path.resolve())
print()

dates_resolved = sum(1 for r in catalogue_rows if r["report_date_iso"])
weeks_resolved = sum(1 for r in catalogue_rows if r["sitrep_week"])
print(f"Rows with a resolved report date: {dates_resolved} / {len(catalogue_rows)}")
print(f"Rows with a resolved sitrep week number: {weeks_resolved} / {len(catalogue_rows)}")
print()

print("Sample of first 3 catalogue rows:")
for row in catalogue_rows[:3]:
    print(row)

INFO | Parsed 484 catalogue rows from 484 candidate table rows.
INFO | Catalogue saved to C:\Users\USER\lassa_fever_nimr\data\interim\ncdc_catalogue.csv
Catalogue rows created: 484
Rows skipped (missing title or URL): 0
Catalogue saved to: C:\Users\USER\lassa_fever_nimr\data\interim\ncdc_catalogue.csv

Rows with a resolved report date: 483 / 484
Rows with a resolved sitrep week number: 484 / 484

Sample of first 3 catalogue rows:
{'report_id': '7dcebbe7227178b3', 'report_title': 'An update of Lassa fever outbreak in Nigeria for Week 36', 'source_url': 'https://www.ncdc.gov.ng/themes/common/files/sitreps/2c311ea5d133f34c14cf42ee191c99d6.pdf', 'catalogue_retrieved_at_utc': '2026-09-28T18:54:54.826339+00:00', 'row_sequence_number': '1', 'sitrep_week': '36', 'report_date_iso': '2026-09-05', 'report_date_source': 'html_comment', 'download_filename': 'An update of Lassa fever outbreak in Nigeria_050926_36.pdf'}
{'report_id': 'cac6b956d006220b', 'report_title': 'An update of Lassa fever outbr

In [14]:
# Phase 2 diagnostic: identify the row with an unresolved report date
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
catalogue_csv_path = PROJECT_ROOT / "data" / "interim" / "ncdc_catalogue.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

unresolved = [r for r in rows if not r["report_date_iso"]]

print("Rows with unresolved report_date_iso:", len(unresolved))
for row in unresolved:
    print(row)

Rows with unresolved report_date_iso: 1
{'report_id': '3f71466b6078bfab', 'report_title': 'An update of Lassa fever outbreak in Nigeria for Week 613566758', 'source_url': 'https://www.ncdc.gov.ng/themes/common/files/sitreps/f781e8e072592b37f51a19723a01cbba.pdf', 'catalogue_retrieved_at_utc': '2026-09-28T18:54:54.826339+00:00', 'row_sequence_number': '484', 'sitrep_week': '613566758', 'report_date_iso': '', 'report_date_source': 'unresolved', 'download_filename': 'An update of Lassa fever outbreak in Nigeria_000000_613566758.pdf'}


In [15]:
# Phase 2 diagnostic: inspect the raw HTML for the malformed row
# This cell is self-contained and imports everything it uses.

from pathlib import Path
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
raw_html_path = PROJECT_ROOT / "data" / "interim" / "ncdc_sitrep_page_raw.html"

html_text = raw_html_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html_text, "html.parser")

target_href_fragment = "f781e8e072592b37f51a19723a01cbba.pdf"

found = False
for tr in soup.find_all("tr"):
    a_tag = tr.find("a", href=True)
    if a_tag and target_href_fragment in a_tag["href"]:
        found = True
        print("Found matching row. Full raw HTML:")
        print(tr.prettify())
        break

if not found:
    print("Row not found by href fragment. Something changed between fetch and now.")

Found matching row. Full raw HTML:
<tr>
 <td>
  346
 </td>
 <td>
  An update of Lassa fever outbreak in Nigeria for Week 3
 </td>
 <!--<td>18 January 2020</td>-->
 <td>
  <a class="black-text" download="An update of Lassa fever outbreak in Nigeria_180120_3.pdf" href="/themes/common/files/sitreps/f781e8e072592b37f51a19723a01cbba.pdf">
   <i class="fa fa-download">
   </i>
  </a>
 </td>
</tr>



In [16]:
# Phase 2 diagnostic: find ALL rows sharing the same PDF href
# This cell is self-contained and imports everything it uses.

from pathlib import Path
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
raw_html_path = PROJECT_ROOT / "data" / "interim" / "ncdc_sitrep_page_raw.html"

html_text = raw_html_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html_text, "html.parser")

target_href_fragment = "f781e8e072592b37f51a19723a01cbba.pdf"

matches = []
for tr in soup.find_all("tr"):
    a_tag = tr.find("a", href=True)
    if a_tag and target_href_fragment in a_tag["href"]:
        matches.append(tr)

print("Total <tr> rows referencing this PDF file:", len(matches))
print()

for i, tr in enumerate(matches, start=1):
    print(f"--- Match {i} ---")
    print(tr.prettify())
    print()

Total <tr> rows referencing this PDF file: 2

--- Match 1 ---
<tr>
 <td>
  346
 </td>
 <td>
  An update of Lassa fever outbreak in Nigeria for Week 3
 </td>
 <!--<td>18 January 2020</td>-->
 <td>
  <a class="black-text" download="An update of Lassa fever outbreak in Nigeria_180120_3.pdf" href="/themes/common/files/sitreps/f781e8e072592b37f51a19723a01cbba.pdf">
   <i class="fa fa-download">
   </i>
  </a>
 </td>
</tr>


--- Match 2 ---
<tr>
 <td>
  484
 </td>
 <td>
  An update of Lassa fever outbreak in Nigeria for Week 613566758
 </td>
 <!--<td></td>-->
 <td>
  <a class="black-text" download="An update of Lassa fever outbreak in Nigeria_000000_613566758.pdf" href="/themes/common/files/sitreps/f781e8e072592b37f51a19723a01cbba.pdf">
   <i class="fa fa-download">
   </i>
  </a>
 </td>
</tr>




In [17]:
# Phase 2 (corrected): build catalogue with deduplication and audit logging
# This cell is self-contained and imports everything it uses.

import re
import csv
import hashlib
import logging
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin

from bs4 import BeautifulSoup, Comment

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

BASE_URL = "https://www.ncdc.gov.ng"
raw_html_path = INTERIM_DIR / "ncdc_sitrep_page_raw.html"

if not raw_html_path.exists():
    raise RuntimeError(
        f"Raw HTML file not found at {raw_html_path}. "
        "Re-run the earlier fetch cell before running this one."
    )

html_text = raw_html_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html_text, "html.parser")

WEEK_PATTERN = re.compile(r"week\s+(\d+)", re.IGNORECASE)
COMMENT_DATE_PATTERN = re.compile(r"<td>\s*(.*?)\s*</td>", re.IGNORECASE)
FILENAME_DATE_WEEK_PATTERN = re.compile(r"_(\d{6})_(\d+)\.pdf$", re.IGNORECASE)

def parse_comment_date(tr_tag):
    for node in tr_tag.find_all(string=lambda t: isinstance(t, Comment)):
        match = COMMENT_DATE_PATTERN.search(str(node))
        if match:
            raw_text = match.group(1).strip()
            if not raw_text:
                return raw_text, None
            for fmt in ("%d %B %Y", "%d %b %Y", "%B %d, %Y"):
                try:
                    return raw_text, datetime.strptime(raw_text, fmt).date().isoformat()
                except ValueError:
                    continue
            return raw_text, None
    return None, None

def parse_filename_date_week(download_attr):
    if not download_attr:
        return None, None
    match = FILENAME_DATE_WEEK_PATTERN.search(download_attr)
    if not match:
        return None, None
    ddmmyy, week = match.group(1), match.group(2)
    if ddmmyy == "000000":
        return None, week
    try:
        parsed_date = datetime.strptime(ddmmyy, "%d%m%y").date().isoformat()
    except ValueError:
        parsed_date = None
    return parsed_date, week

rows_with_pdf = []
for tr in soup.find_all("tr"):
    a_tag = tr.find("a", href=True)
    if a_tag and a_tag["href"].strip().lower().endswith(".pdf"):
        rows_with_pdf.append((tr, a_tag))

parsed_rows = []
retrieved_at = datetime.now(timezone.utc).isoformat()
parse_failures = []

for tr, a_tag in rows_with_pdf:
    cells = tr.find_all(["td", "th"])
    row_sequence_number = cells[0].get_text(strip=True) if len(cells) > 0 else ""
    report_title = cells[1].get_text(strip=True) if len(cells) > 1 else ""

    href = a_tag["href"].strip()
    source_url = urljoin(BASE_URL, href)
    download_filename = a_tag.get("download", "").strip()

    week_match = WEEK_PATTERN.search(report_title)
    sitrep_week = week_match.group(1) if week_match else None

    comment_date_raw, comment_date_iso = parse_comment_date(tr)
    filename_date_iso, filename_week = parse_filename_date_week(download_filename)

    best_date_iso = comment_date_iso or filename_date_iso

    report_id = hashlib.sha256(source_url.encode("utf-8")).hexdigest()[:16]

    if not report_title or not source_url:
        parse_failures.append({"row_sequence_number": row_sequence_number, "href": href})
        continue

    parsed_rows.append({
        "report_id": report_id,
        "report_title": report_title,
        "source_url": source_url,
        "catalogue_retrieved_at_utc": retrieved_at,
        "row_sequence_number": row_sequence_number,
        "sitrep_week": sitrep_week,
        "report_date_iso": best_date_iso,
        "report_date_source": "html_comment" if comment_date_iso else (
            "filename" if filename_date_iso else "unresolved"
        ),
        "download_filename": download_filename,
    })

# ---------------------------------------------------------------------------
# Deduplicate by source_url. Some NCDC table entries point to the same PDF
# more than once (observed: a corrupted placeholder row duplicating a valid
# earlier entry). When duplicates exist, keep the row with a resolved date;
# if none have a resolved date, keep the first occurrence. Every excluded
# duplicate is logged explicitly for audit purposes.
# ---------------------------------------------------------------------------

by_url = {}
for row in parsed_rows:
    url = row["source_url"]
    by_url.setdefault(url, []).append(row)

catalogue_rows = []
excluded_duplicates = []

for url, group in by_url.items():
    if len(group) == 1:
        catalogue_rows.append(group[0])
        continue

    resolved = [r for r in group if r["report_date_iso"]]
    kept = resolved[0] if resolved else group[0]
    catalogue_rows.append(kept)

    for row in group:
        if row is not kept:
            excluded_duplicates.append(row)

    logger.warning(
        "Deduplicated %d rows for source_url %s. Kept report_id %s, excluded %d.",
        len(group), url, kept["report_id"], len(group) - 1
    )

catalogue_rows.sort(key=lambda r: int(r["row_sequence_number"]))

logger.info("Parsed %d rows, %d unique after deduplication, %d duplicates excluded.",
            len(parsed_rows), len(catalogue_rows), len(excluded_duplicates))
if parse_failures:
    logger.warning("Skipped %d rows due to missing title or URL.", len(parse_failures))

if len(catalogue_rows) == 0:
    raise RuntimeError(
        "Catalogue is empty after parsing. Stopping before the download stage."
    )

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
fieldnames = list(catalogue_rows[0].keys())
with catalogue_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(catalogue_rows)

if excluded_duplicates:
    duplicates_csv_path = INTERIM_DIR / "ncdc_catalogue_excluded_duplicates.csv"
    with duplicates_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(excluded_duplicates)
    logger.info("Excluded duplicate rows saved to %s", duplicates_csv_path)

logger.info("Final catalogue saved to %s", catalogue_csv_path)

print("Unique catalogue rows created:", len(catalogue_rows))
print("Duplicate rows excluded:", len(excluded_duplicates))
print("Rows skipped (missing title or URL):", len(parse_failures))
print("Catalogue saved to:", catalogue_csv_path.resolve())
if excluded_duplicates:
    print("Excluded duplicates saved to:", (INTERIM_DIR / 'ncdc_catalogue_excluded_duplicates.csv').resolve())
print()

dates_resolved = sum(1 for r in catalogue_rows if r["report_date_iso"])
weeks_resolved = sum(1 for r in catalogue_rows if r["sitrep_week"])
print(f"Rows with a resolved report date: {dates_resolved} / {len(catalogue_rows)}")
print(f"Rows with a resolved sitrep week number: {weeks_resolved} / {len(catalogue_rows)}")

WARNING | Deduplicated 2 rows for source_url https://www.ncdc.gov.ng/themes/common/files/sitreps/ea1bb56e795d90f28c78d38cfc3b2284.pdf. Kept report_id d6caaab5c495542b, excluded 1.
WARNING | Deduplicated 2 rows for source_url https://www.ncdc.gov.ng/themes/common/files/sitreps/f781e8e072592b37f51a19723a01cbba.pdf. Kept report_id 3f71466b6078bfab, excluded 1.
INFO | Parsed 484 rows, 482 unique after deduplication, 2 duplicates excluded.
INFO | Excluded duplicate rows saved to C:\Users\USER\lassa_fever_nimr\data\interim\ncdc_catalogue_excluded_duplicates.csv
INFO | Final catalogue saved to C:\Users\USER\lassa_fever_nimr\data\interim\ncdc_catalogue.csv
Unique catalogue rows created: 482
Duplicate rows excluded: 2
Rows skipped (missing title or URL): 0
Catalogue saved to: C:\Users\USER\lassa_fever_nimr\data\interim\ncdc_catalogue.csv
Excluded duplicates saved to: C:\Users\USER\lassa_fever_nimr\data\interim\ncdc_catalogue_excluded_duplicates.csv

Rows with a resolved report date: 482 / 482
R

In [18]:
# Phase 3: Download NCDC report PDFs
# This cell is self-contained and imports everything it uses.

import csv
import time
import hashlib
import logging
import re
from pathlib import Path
from datetime import datetime, timezone

import requests

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
RAW_REPORTS_DIR = PROJECT_ROOT / "data" / "raw" / "ncdc_reports"
LOGS_DIR = PROJECT_ROOT / "logs"
RAW_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,*/*;q=0.8",
}

REQUEST_TIMEOUT = 30
MAX_RETRIES = 4
RETRY_DELAY_SECONDS = 5
DELAY_BETWEEN_DOWNLOADS_SECONDS = 1.5

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
if not catalogue_csv_path.exists():
    raise RuntimeError(
        f"Catalogue not found at {catalogue_csv_path}. "
        "Run Phase 2 before starting downloads."
    )

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

if len(catalogue_rows) == 0:
    raise RuntimeError("Catalogue is empty. Stopping before downloading anything.")

logger.info("Loaded %d catalogue rows for download.", len(catalogue_rows))

def safe_filename(report_id, week, date_iso, title):
    """Build a safe, unique, human-readable filename."""
    slug = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")[:60]
    week_part = f"wk{week}" if week else "wkunknown"
    date_part = date_iso if date_iso else "datenknown"
    return f"{date_part}_{week_part}_{report_id}_{slug}.pdf"

def sha256_of_file(path):
    hasher = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def fetch_pdf_with_retries(url, headers, timeout, max_retries, retry_delay):
    last_exception = None
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            if response.status_code == 200:
                return response, None
            if response.status_code == 500:
                logger.warning("HTTP 500 on attempt %d for %s. Retrying.", attempt, url)
                time.sleep(retry_delay)
                continue
            logger.warning("Status %d on attempt %d for %s.", response.status_code, attempt, url)
            time.sleep(retry_delay)
        except requests.RequestException as exc:
            last_exception = exc
            logger.warning("Request exception on attempt %d for %s: %s", attempt, url, exc)
            time.sleep(retry_delay)
    return None, (str(last_exception) if last_exception else "Non-200 status after retries")

# ---------------------------------------------------------------------------
# Load existing acquisition log for resumability, if present
# ---------------------------------------------------------------------------

acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
already_downloaded_ids = set()

if acquisition_log_path.exists():
    with acquisition_log_path.open("r", encoding="utf-8") as f:
        existing_log = list(csv.DictReader(f))
    for row in existing_log:
        if row.get("download_status") == "success":
            local_path = Path(row.get("local_file", ""))
            if local_path.exists():
                already_downloaded_ids.add(row["report_id"])
    logger.info("Found existing acquisition log with %d prior successful downloads.",
                len(already_downloaded_ids))
else:
    existing_log = []

log_fieldnames = [
    "report_id", "report_title", "source_url", "resolved_file_url",
    "local_file", "content_type", "sha256_hash", "download_status",
    "error_message", "downloaded_at_utc",
]

new_log_entries = []
success_count = 0
skip_count = 0
fail_count = 0

for i, row in enumerate(catalogue_rows, start=1):
    report_id = row["report_id"]
    report_title = row["report_title"]
    source_url = row["source_url"]
    week = row.get("sitrep_week", "")
    date_iso = row.get("report_date_iso", "")

    if report_id in already_downloaded_ids:
        skip_count += 1
        continue

    filename = safe_filename(report_id, week, date_iso, report_title)
    local_path = RAW_REPORTS_DIR / filename

    if local_path.exists():
        logger.info("File already exists on disk for %s, skipping re-download: %s",
                    report_id, local_path)
        file_hash = sha256_of_file(local_path)
        new_log_entries.append({
            "report_id": report_id,
            "report_title": report_title,
            "source_url": source_url,
            "resolved_file_url": source_url,
            "local_file": str(local_path),
            "content_type": "application/pdf",
            "sha256_hash": file_hash,
            "download_status": "success",
            "error_message": "",
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        })
        success_count += 1
        continue

    logger.info("Downloading (%d/%d) report_id %s from %s", i, len(catalogue_rows), report_id, source_url)
    response, error_message = fetch_pdf_with_retries(
        source_url, HEADERS, REQUEST_TIMEOUT, MAX_RETRIES, RETRY_DELAY_SECONDS
    )

    if response is None:
        logger.error("Failed to download report_id %s: %s", report_id, error_message)
        new_log_entries.append({
            "report_id": report_id,
            "report_title": report_title,
            "source_url": source_url,
            "resolved_file_url": "",
            "local_file": "",
            "content_type": "",
            "sha256_hash": "",
            "download_status": "failed",
            "error_message": error_message,
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        })
        fail_count += 1
        time.sleep(DELAY_BETWEEN_DOWNLOADS_SECONDS)
        continue

    content_type = response.headers.get("Content-Type", "")
    local_path.write_bytes(response.content)
    file_hash = sha256_of_file(local_path)

    new_log_entries.append({
        "report_id": report_id,
        "report_title": report_title,
        "source_url": source_url,
        "resolved_file_url": response.url,
        "local_file": str(local_path),
        "content_type": content_type,
        "sha256_hash": file_hash,
        "download_status": "success",
        "error_message": "",
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    })
    success_count += 1

    time.sleep(DELAY_BETWEEN_DOWNLOADS_SECONDS)

# ---------------------------------------------------------------------------
# Merge with any existing log and save
# ---------------------------------------------------------------------------

combined_log = {row["report_id"]: row for row in existing_log}
for row in new_log_entries:
    combined_log[row["report_id"]] = row

with acquisition_log_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=log_fieldnames)
    writer.writeheader()
    writer.writerows(combined_log.values())

logger.info("Acquisition log saved to %s", acquisition_log_path)

print("Newly downloaded this run:", success_count - skip_count if success_count >= skip_count else success_count)
print("Skipped (already on disk from a prior run):", skip_count)
print("Failed this run:", fail_count)
print("Total entries in acquisition log:", len(combined_log))
print("Acquisition log saved to:", acquisition_log_path.resolve())

INFO | Loaded 482 catalogue rows for download.
INFO | Found existing acquisition log with 477 prior successful downloads.
INFO | Downloading (1/482) report_id 7dcebbe7227178b3 from https://www.ncdc.gov.ng/themes/common/files/sitreps/2c311ea5d133f34c14cf42ee191c99d6.pdf
INFO | Downloading (2/482) report_id cac6b956d006220b from https://www.ncdc.gov.ng/themes/common/files/sitreps/c657ffd9e1497fd496235d98f0bc029a.pdf
INFO | Downloading (154/482) report_id 9041da84d8d104bc from https://www.ncdc.gov.ng/themes/common/files/sitreps/7bff918b81a8e610200b879d929995da.pdf
WARNING | Status 404 on attempt 1 for https://www.ncdc.gov.ng/themes/common/files/sitreps/7bff918b81a8e610200b879d929995da.pdf.
WARNING | Status 404 on attempt 2 for https://www.ncdc.gov.ng/themes/common/files/sitreps/7bff918b81a8e610200b879d929995da.pdf.


KeyboardInterrupt: 

In [ ]:
# Diagnostic: look up specific rows by row_sequence_number and check their download status
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
catalogue_csv_path = PROJECT_ROOT / "data" / "interim" / "ncdc_catalogue.csv"
acquisition_log_path = PROJECT_ROOT / "data" / "interim" / "acquisition_log.csv"

target_sequence_numbers = {"152", "220", "269"}

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

for row in catalogue_rows:
    if row["row_sequence_number"] in target_sequence_numbers:
        log_entry = log_rows.get(row["report_id"], {})
        print(f"Row {row['row_sequence_number']}:")
        print("  Title:", row["report_title"])
        print("  Sitrep week:", row["sitrep_week"])
        print("  Report date:", row["report_date_iso"])
        print("  Report ID:", row["report_id"])
        print("  Download status:", log_entry.get("download_status", "not in log"))
        print("  Error message:", log_entry.get("error_message", ""))
        print()

In [ ]:
# Diagnostic: verify the three previously-failed reports are genuinely on disk
# This cell is self-contained and imports everything it uses.

import csv
import hashlib
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
acquisition_log_path = PROJECT_ROOT / "data" / "interim" / "acquisition_log.csv"

target_report_ids = {"05a6a45aca15b1ac", "f2c55a6e97a5e581", "0c0d60e6b8230bba"}

def sha256_of_file(path):
    hasher = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

for report_id in target_report_ids:
    entry = log_rows.get(report_id)
    if entry is None:
        print(f"{report_id}: not found in acquisition log at all")
        continue

    local_path = Path(entry["local_file"])
    exists = local_path.exists()
    size_bytes = local_path.stat().st_size if exists else 0
    hash_matches = None
    if exists:
        hash_matches = (sha256_of_file(local_path) == entry["sha256_hash"])

    print(f"Report {report_id}:")
    print("  Local file:", local_path)
    print("  File exists on disk:", exists)
    print("  File size (bytes):", size_bytes)
    print("  Hash matches log:", hash_matches)
    print()

In [ ]:
pip install pymupdf

In [ ]:
# Phase 4, Step 1: inspect raw extracted text from sample PDFs across the date range
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import pymupdf

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
acquisition_log_path = PROJECT_ROOT / "data" / "interim" / "acquisition_log.csv"
catalogue_csv_path = PROJECT_ROOT / "data" / "interim" / "ncdc_catalogue.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

# Only consider rows that downloaded successfully and have a resolved date
usable_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
]
usable_rows.sort(key=lambda r: r["report_date_iso"])

# Pick three samples spread across the date range: earliest, middle, latest
sample_indices = [0, len(usable_rows) // 2, len(usable_rows) - 1]
sample_rows = [usable_rows[i] for i in sample_indices]

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text_parts = [page.get_text() for page in doc]
    doc.close()
    return "\n".join(text_parts)

for row in sample_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    print("=" * 80)
    print("Report title:", row["report_title"])
    print("Report date:", row["report_date_iso"])
    print("Sitrep week:", row["sitrep_week"])
    print("File:", local_path.name)
    print("=" * 80)
    text = extract_text(local_path)
    print(text[:3000])
    print()
    print("... [truncated] ...")
    print()

In [ ]:
# Phase 4, Step 2 (audit): verify every catalogue date/week against the date
# actually printed inside each PDF. This must run before any case-number
# extraction, since a wrong month/year would silently corrupt aggregation.
# This cell is self-contained and imports everything it uses.

import re
import csv
import logging
from pathlib import Path

import pymupdf

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

# ---------------------------------------------------------------------------
# Patterns to find the report's own stated Epi Week and year inside the PDF
# text, covering both the "EpiWeek:522021" (no-space, older/newer templates)
# and "Epi Week: 34 2026" (spaced) styles seen so far, plus the older
# "Situation Report No. 11 - March 31, 2017" narrative style.
# ---------------------------------------------------------------------------

EPI_WEEK_COMPACT = re.compile(r"Epi\s*Week:?\s*(\d{1,2})\s*(20\d{2})", re.IGNORECASE)
NARRATIVE_DATE = re.compile(
    r"(January|February|March|April|May|June|July|August|September|October|November|December)"
    r"\s+\d{1,2},?\s+(20\d{2})",
    re.IGNORECASE,
)

def extract_text(pdf_path, max_chars=4000):
    doc = fitz.open(pdf_path)
    text_parts = []
    total_len = 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

audit_rows = []
mismatches = []
unresolved_in_pdf = []

usable_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
]

logger.info("Auditing dates for %d successfully downloaded reports.", len(usable_rows))

for i, row in enumerate(usable_rows, start=1):
    report_id = row["report_id"]
    local_path = Path(log_rows[report_id]["local_file"])
    catalogue_week = row["sitrep_week"]
    catalogue_year = row["report_date_iso"][:4] if row["report_date_iso"] else None

    text = extract_text(local_path)

    pdf_week, pdf_year, match_source = None, None, None

    compact_match = EPI_WEEK_COMPACT.search(text)
    if compact_match:
        pdf_week, pdf_year = compact_match.group(1), compact_match.group(2)
        match_source = "epi_week_field"
    else:
        narrative_match = NARRATIVE_DATE.search(text)
        if narrative_match:
            pdf_year = narrative_match.group(2)
            match_source = "narrative_date"

    year_match = (pdf_year == catalogue_year) if pdf_year else None
    week_match = (pdf_week == catalogue_week) if pdf_week else None

    status = "unresolved_in_pdf"
    if pdf_year is None:
        unresolved_in_pdf.append(row)
    elif year_match is False or (week_match is False):
        status = "mismatch"
        mismatches.append({
            "report_id": report_id,
            "report_title": row["report_title"],
            "local_file": str(local_path),
            "catalogue_week": catalogue_week,
            "catalogue_year": catalogue_year,
            "pdf_week": pdf_week,
            "pdf_year": pdf_year,
            "match_source": match_source,
        })
    elif year_match is True:
        status = "confirmed"

    audit_rows.append({
        "report_id": report_id,
        "report_title": row["report_title"],
        "catalogue_week": catalogue_week,
        "catalogue_year": catalogue_year,
        "pdf_week": pdf_week,
        "pdf_year": pdf_year,
        "match_source": match_source,
        "audit_status": status,
    })

    if i % 100 == 0:
        logger.info("Audited %d / %d reports so far.", i, len(usable_rows))

audit_csv_path = INTERIM_DIR / "date_audit.csv"
with audit_csv_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(audit_rows[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(audit_rows)

confirmed_count = sum(1 for r in audit_rows if r["audit_status"] == "confirmed")
mismatch_count = sum(1 for r in audit_rows if r["audit_status"] == "mismatch")
unresolved_count = sum(1 for r in audit_rows if r["audit_status"] == "unresolved_in_pdf")

logger.info("Date audit complete. Confirmed: %d, Mismatch: %d, Unresolved: %d",
            confirmed_count, mismatch_count, unresolved_count)

print("Total reports audited:", len(audit_rows))
print("Confirmed (catalogue matches PDF content):", confirmed_count)
print("Mismatches (catalogue disagrees with PDF content):", mismatch_count)
print("Unresolved (could not find a year in the PDF text at all):", unresolved_count)
print()
print("Full audit saved to:", audit_csv_path.resolve())
print()

if mismatches:
    print("Sample mismatches (up to 10):")
    for m in mismatches[:10]:
        print(m)

In [ ]:
# Phase 4, Step 2b (corrected): re-audit with numeric week comparison, and
# inspect what the regex is actually matching for the suspicious '13' cases.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

audit_csv_path = INTERIM_DIR / "date_audit.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with audit_csv_path.open("r", encoding="utf-8") as f:
    audit_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

def normalize_week(value):
    if value is None or value == "":
        return None
    try:
        return int(value)
    except ValueError:
        return None

true_mismatches = []
false_positive_padding = []

for row in audit_rows:
    if row["audit_status"] != "mismatch":
        continue
    cat_week_num = normalize_week(row["catalogue_week"])
    pdf_week_num = normalize_week(row["pdf_week"])
    year_equal = (row["catalogue_year"] == row["pdf_year"])
    week_equal = (cat_week_num == pdf_week_num) if (cat_week_num is not None and pdf_week_num is not None) else False

    if year_equal and week_equal:
        false_positive_padding.append(row)
    else:
        true_mismatches.append(row)

print("Originally flagged mismatches:", sum(1 for r in audit_rows if r["audit_status"] == "mismatch"))
print("Of those, false positives from zero-padding only:", len(false_positive_padding))
print("Of those, genuine mismatches remaining:", len(true_mismatches))
print()

EPI_WEEK_COMPACT = re.compile(r"Epi\s*Week:?\s*(\d{1,2})\s*(20\d{2})", re.IGNORECASE)

def extract_text(pdf_path, max_chars=6000):
    doc = fitz.open(pdf_path)
    text_parts = []
    total_len = 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

suspicious_13_cases = [r for r in true_mismatches if r["pdf_week"] == "13"]
print(f"Genuine mismatches where pdf_week == '13': {len(suspicious_13_cases)}")
print()

for row in suspicious_13_cases[:3]:
    report_id = row["report_id"]
    log_entry = log_rows.get(report_id)

    print("=" * 80)
    print("Report title:", row["report_title"])
    print("Catalogue week/year:", row["catalogue_week"], row["catalogue_year"])

    if log_entry is None:
        print("No acquisition log entry found for this report_id. Skipping.")
        print()
        continue

    local_path = Path(log_entry["local_file"])
    print("Local file:", local_path)

    if not local_path.exists():
        print("File does not exist on disk. Skipping.")
        print()
        continue

    text = extract_text(local_path)
    match = EPI_WEEK_COMPACT.search(text)
    if match:
        start = max(0, match.start() - 150)
        end = min(len(text), match.end() + 150)
        print("Context around matched text:")
        print(repr(text[start:end]))
    else:
        print("No match found on re-run (unexpected).")
    print()

print("Remaining genuine, non-'13' mismatches:")
for row in true_mismatches:
    if row["pdf_week"] != "13":
        print(row)

In [ ]:
# Phase 4, Step 2c: robust week/year extraction using three independent
# signals per report, with cross-validation, to separate real catalogue
# errors from stale-header artifacts inside NCDC's own PDFs.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

# ---------------------------------------------------------------------------
# Three independent signals:
#   A. Figure 1 caption: "epidemiological week NN, YYYY" (spaced or compact)
#   B. Table 1 label: "Current week (week NN)" or "Currentweek(weekNN)"
#   C. Running header field: "Epi Week: NN YYYY" (known to be unreliable
#      on later pages of a small number of reports)
# ---------------------------------------------------------------------------

CAPTION_PATTERN = re.compile(
    r"epidemiological\s*week\s*(\d{1,2})\D{0,15}?(20\d{2})", re.IGNORECASE
)
CURRENT_WEEK_PATTERN = re.compile(
    r"current\s*week\D{0,10}?week\s*(\d{1,2})", re.IGNORECASE
)
HEADER_PATTERN = re.compile(r"Epi\s*Week:?\s*(\d{1,2})\s*(20\d{2})", re.IGNORECASE)

def normalize_week(value):
    if value in (None, ""):
        return None
    try:
        return int(value)
    except ValueError:
        return None

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

usable_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
]

results = []
for i, row in enumerate(usable_rows, start=1):
    report_id = row["report_id"]
    local_path = Path(log_rows[report_id]["local_file"])
    catalogue_week = normalize_week(row["sitrep_week"])
    catalogue_year = row["report_date_iso"][:4] if row["report_date_iso"] else None

    text = extract_text(local_path)

    caption_match = CAPTION_PATTERN.search(text)
    caption_week = normalize_week(caption_match.group(1)) if caption_match else None
    caption_year = caption_match.group(2) if caption_match else None

    current_week_match = CURRENT_WEEK_PATTERN.search(text)
    current_week = normalize_week(current_week_match.group(1)) if current_week_match else None

    header_match = HEADER_PATTERN.search(text)
    header_week = normalize_week(header_match.group(1)) if header_match else None
    header_year = header_match.group(2) if header_match else None

    signals_agreeing_with_catalogue_week = sum(
        1 for w in (caption_week, current_week, header_week)
        if w is not None and w == catalogue_week
    )

    year_signals = [y for y in (caption_year, header_year) if y is not None]
    year_agrees = catalogue_year in year_signals if year_signals else None

    if signals_agreeing_with_catalogue_week >= 2 and (year_agrees is not False):
        status = "confirmed"
    elif signals_agreeing_with_catalogue_week >= 1 and (year_agrees is not False):
        status = "likely_ok"
    else:
        status = "needs_review"

    results.append({
        "report_id": report_id,
        "report_title": row["report_title"],
        "local_file": str(local_path),
        "catalogue_week": catalogue_week,
        "catalogue_year": catalogue_year,
        "caption_week": caption_week,
        "caption_year": caption_year,
        "current_week_field": current_week,
        "header_week": header_week,
        "header_year": header_year,
        "status": status,
    })

    if i % 100 == 0:
        print(f"Processed {i} / {len(usable_rows)}")

results_csv_path = INTERIM_DIR / "date_audit_v2.csv"
with results_csv_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(results[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

confirmed = sum(1 for r in results if r["status"] == "confirmed")
likely_ok = sum(1 for r in results if r["status"] == "likely_ok")
needs_review = sum(1 for r in results if r["status"] == "needs_review")

print()
print("Total reports:", len(results))
print("Confirmed (2+ signals agree, no year conflict):", confirmed)
print("Likely OK (1 signal agrees, no year conflict):", likely_ok)
print("Needs review:", needs_review)
print()
print("Full results saved to:", results_csv_path.resolve())
print()
print("Rows needing review:")
for r in results:
    if r["status"] == "needs_review":
        print(r)

In [ ]:
# Phase 4, Step 2d: isolate needs_review rows that fall inside the 2020-2025
# training window and inspect their Table 1 area directly.
# This cell is self-contained and imports everything it uses.

import csv
from datetime import date
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
audit_v2_csv_path = INTERIM_DIR / "date_audit_v2.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_by_id = {row["report_id"]: row for row in csv.DictReader(f)}

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

with audit_v2_csv_path.open("r", encoding="utf-8") as f:
    audit_rows = list(csv.DictReader(f))

TRAIN_START = date(2020, 1, 1)
TRAIN_END = date(2025, 12, 31)

def in_training_window(report_id):
    cat_row = catalogue_by_id.get(report_id)
    if not cat_row or not cat_row.get("report_date_iso"):
        return False
    try:
        d = date.fromisoformat(cat_row["report_date_iso"])
    except ValueError:
        return False
    return TRAIN_START <= d <= TRAIN_END

in_window_review = [
    r for r in audit_rows
    if r["status"] == "needs_review" and in_training_window(r["report_id"])
]

print(f"needs_review rows inside 2020-2025 training window: {len(in_window_review)}")
print()

def extract_text(pdf_path, max_chars=3000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

for row in in_window_review:
    report_id = row["report_id"]
    log_entry = log_rows.get(report_id)
    print("=" * 80)
    print("Report title:", row["report_title"])
    print("Catalogue week/year:", row["catalogue_week"], row["catalogue_year"])
    print("Caption week/year found:", row["caption_week"], row["caption_year"])
    print("Current-week field found:", row["current_week_field"])
    print("Header week/year found:", row["header_week"], row["header_year"])

    if log_entry is None:
        print("No acquisition log entry found.")
        print()
        continue

    local_path = Path(log_entry["local_file"])
    print("File:", local_path.name)
    text = extract_text(local_path)
    print("First 1500 characters of extracted text:")
    print(text[:1500])
    print()

In [ ]:
# Phase 4, Step 2e: diagnose the three empty-text PDFs, and write the
# manual corrections log for the two confirmed catalogue errors.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path
from datetime import datetime, timezone

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

empty_text_report_ids = [
    "f67b8cccc71e6258",  # Week 30, 2021
    "c92f74f81aca205d",  # Week 23, 2021
    "f80888274a287aa3",  # Week 21, 2021
]

print("Diagnosing suspected image-only PDFs:")
print()
for report_id in empty_text_report_ids:
    local_path = Path(log_rows[report_id]["local_file"])
    doc = fitz.open(local_path)
    page_count = len(doc)
    total_text_chars = sum(len(page.get_text()) for page in doc)
    total_images = sum(len(page.get_images(full=True)) for page in doc)
    doc.close()

    print(f"File: {local_path.name}")
    print(f"  Page count: {page_count}")
    print(f"  Total extractable text characters: {total_text_chars}")
    print(f"  Total embedded images across all pages: {total_images}")
    print(f"  Likely scanned/image-only: {total_text_chars < 20 and total_images > 0}")
    print()

# ---------------------------------------------------------------------------
# Manual corrections log: records the two confirmed catalogue errors found
# during date audit, with justification, for the audit trail.
# ---------------------------------------------------------------------------

corrections_csv_path = INTERIM_DIR / "manual_catalogue_corrections.csv"
now = datetime.now(timezone.utc).isoformat()

corrections = [
    {
        "report_id": "e9bede3874ffce7a",
        "report_title_original": "An update of Lassa fever outbreak in Nigeria for Week 53",
        "correction_type": "date_and_week_correction",
        "original_sitrep_week": "53",
        "original_report_date_iso": "2022-12-29",
        "corrected_sitrep_week": "1",
        "corrected_report_date_iso": "2023-01-02",
        "justification": (
            "PDF content states 'Lassa fever Situation Report Epi Week 1: 2-8 "
            "January 2023', 'Table 1: Summary of current week (1), cumulative "
            "Epi week 1, 2023'. NCDC site title of 'Week 53' does not match "
            "report content. Verified by direct PDF text inspection."
        ),
        "reviewer_initials": "",
        "review_date": now,
    },
    {
        "report_id": "f2c55a6e97a5e581",
        "report_title_original": "An update of Lassa fever outbreak in Nigeria for Week 23",
        "correction_type": "exclude_wrong_disease",
        "original_sitrep_week": "23",
        "original_report_date_iso": "2022-06-02",
        "corrected_sitrep_week": "",
        "corrected_report_date_iso": "",
        "justification": (
            "PDF content is a Yellow Fever Situation Report (Serial Number 003, "
            "May 2022 Monthly Sitrep), not a Lassa fever weekly sitrep. NCDC "
            "table linked the wrong file to this row. Excluded from dataset; "
            "contains no Lassa fever weekly case data."
        ),
        "reviewer_initials": "",
        "review_date": now,
    },
]

with corrections_csv_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(corrections[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(corrections)

print("Manual corrections log saved to:", corrections_csv_path.resolve())
print(f"Total corrections logged: {len(corrections)}")

In [ ]:
# Phase 4, Step 2g: verify the re-downloaded PDFs are now readable
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

report_ids_to_verify = [
    "f67b8cccc71e6258",
    "c92f74f81aca205d",
    "f80888274a287aa3",
]

for report_id in report_ids_to_verify:
    local_path = Path(log_rows[report_id]["local_file"])
    print(f"Report {report_id}:")
    print("  File:", local_path.name)
    print("  Size (bytes):", local_path.stat().st_size)

    try:
        doc = fitz.open(local_path)
        page_count = len(doc)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        print("  Page count:", page_count)
        print("  Extractable text characters:", len(text))
        print("  First 300 characters of text:")
        print(" ", repr(text[:300]))
    except Exception as exc:
        print("  Failed to open/parse with PyMuPDF:", exc)
    print()

In [ ]:
# Phase 4, Step 2f (corrected): re-download the three corrupted PDFs with
# safer control flow. This cell is self-contained and imports everything it uses.

import csv
import time
import hashlib
import logging
from pathlib import Path
from datetime import datetime, timezone

import requests

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_by_id = {row["report_id"]: row for row in csv.DictReader(f)}

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows_list = list(csv.DictReader(f))
    log_rows = {row["report_id"]: row for row in log_rows_list}

corrupted_report_ids = [
    "f67b8cccc71e6258",
    "c92f74f81aca205d",
    "f80888274a287aa3",
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,*/*;q=0.8",
}
REQUEST_TIMEOUT = 30
MAX_RETRIES = 4
RETRY_DELAY_SECONDS = 5

def sha256_of_file(path):
    hasher = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def attempt_redownload(report_id, source_url, local_path):
    """Returns a dict describing the outcome. Never raises past this point."""
    outcome = {
        "report_id": report_id,
        "download_status": "failed",
        "error_message": "",
        "sha256_hash": "",
        "content_type": "",
        "resolved_file_url": "",
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    response = None
    last_exception = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(source_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if response.status_code == 200:
                break
            logger.warning("Status %d on attempt %d for %s", response.status_code, attempt, source_url)
            response = None
            time.sleep(RETRY_DELAY_SECONDS)
        except requests.RequestException as exc:
            last_exception = exc
            logger.warning("Request exception on attempt %d for %s: %s", attempt, source_url, exc)
            time.sleep(RETRY_DELAY_SECONDS)

    if response is None:
        outcome["error_message"] = f"Re-download failed after retries: {last_exception}"
        return outcome

    content_bytes = response.content
    outcome["content_type"] = response.headers.get("Content-Type", "")
    outcome["resolved_file_url"] = response.url

    if not content_bytes.startswith(b"%PDF"):
        outcome["error_message"] = (
            f"Response does not start with %PDF marker "
            f"(Content-Type: {outcome['content_type']}, size: {len(content_bytes)} bytes)"
        )
        return outcome

    local_path.write_bytes(content_bytes)
    outcome["sha256_hash"] = sha256_of_file(local_path)
    outcome["download_status"] = "success"
    return outcome

updated_log_rows = {row["report_id"]: dict(row) for row in log_rows_list}

for report_id in corrupted_report_ids:
    source_url = catalogue_by_id[report_id]["source_url"]
    local_path = Path(log_rows[report_id]["local_file"])

    print(f"Re-downloading report_id {report_id} from {source_url}")
    result = attempt_redownload(report_id, source_url, local_path)

    print("  Status:", result["download_status"])
    if result["download_status"] == "success":
        print("  New hash:", result["sha256_hash"])
    else:
        print("  Error:", result["error_message"])
    print()

    updated_log_rows[report_id]["download_status"] = result["download_status"]
    updated_log_rows[report_id]["error_message"] = result["error_message"]
    if result["sha256_hash"]:
        updated_log_rows[report_id]["sha256_hash"] = result["sha256_hash"]
    if result["content_type"]:
        updated_log_rows[report_id]["content_type"] = result["content_type"]
    if result["resolved_file_url"]:
        updated_log_rows[report_id]["resolved_file_url"] = result["resolved_file_url"]
    updated_log_rows[report_id]["downloaded_at_utc"] = result["downloaded_at_utc"]

    time.sleep(1.5)

log_fieldnames = list(log_rows_list[0].keys())
with acquisition_log_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=log_fieldnames)
    writer.writeheader()
    writer.writerows(updated_log_rows.values())

print("Acquisition log updated and saved.")

In [ ]:
# Phase 4, Step 2g: verify the re-downloaded PDFs are now readable
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import pymupdf

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

report_ids_to_verify = [
    "f67b8cccc71e6258",
    "c92f74f81aca205d",
    "f80888274a287aa3",
]

for report_id in report_ids_to_verify:
    local_path = Path(log_rows[report_id]["local_file"])
    print(f"Report {report_id}:")
    print("  File:", local_path.name)
    print("  Size (bytes):", local_path.stat().st_size)

    try:
        doc = fitz.open(local_path)
        page_count = len(doc)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        print("  Page count:", page_count)
        print("  Extractable text characters:", len(text))
        print("  First 300 characters of text:")
        print(" ", repr(text[:300]))
    except Exception as exc:
        print("  Failed to open/parse with PyMuPDF:", exc)
    print()

In [ ]:
pip install pikepdf

In [ ]:
# Phase 4, Step 2h: attempt structural repair of the three unreadable PDFs
# using pikepdf, then re-verify with PyMuPDF.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import pikepdf
import pymupdf

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
RAW_REPORTS_DIR = PROJECT_ROOT / "data" / "raw" / "ncdc_reports"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

report_ids_to_repair = [
    "f67b8cccc71e6258",
    "c92f74f81aca205d",
    "f80888274a287aa3",
]

REPAIRED_DIR = RAW_REPORTS_DIR / "repaired"
REPAIRED_DIR.mkdir(parents=True, exist_ok=True)

for report_id in report_ids_to_repair:
    original_path = Path(log_rows[report_id]["local_file"])
    repaired_path = REPAIRED_DIR / original_path.name

    print(f"Report {report_id}: {original_path.name}")

    try:
        with pikepdf.open(original_path) as pdf:
            page_count_pikepdf = len(pdf.pages)
            pdf.save(repaired_path)
        print(f"  pikepdf opened successfully. Page count per pikepdf: {page_count_pikepdf}")
        print(f"  Repaired copy saved to: {repaired_path}")
    except Exception as exc:
        print(f"  pikepdf FAILED to open/repair this file: {exc}")
        print()
        continue

    try:
        doc = fitz.open(repaired_path)
        page_count = len(doc)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        print(f"  PyMuPDF on repaired file - page count: {page_count}, text characters: {len(text)}")
        if text.strip():
            print("  First 300 characters of text:")
            print(" ", repr(text[:300]))
    except Exception as exc:
        print(f"  PyMuPDF still failed on repaired file: {exc}")

    print()

In [ ]:
# Phase 4, Step 2i: finalize the three unrecoverable PDFs in the manual
# corrections log, then close out date validation for Phase 4.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
corrections_csv_path = INTERIM_DIR / "manual_catalogue_corrections.csv"

with corrections_csv_path.open("r", encoding="utf-8") as f:
    existing_corrections = list(csv.DictReader(f))
    fieldnames = list(existing_corrections[0].keys())

now = datetime.now(timezone.utc).isoformat()

unrecoverable_report_ids = {
    "f67b8cccc71e6258": ("30", "2021-07-23"),
    "c92f74f81aca205d": ("23", "2021-06-04"),
    "f80888274a287aa3": ("21", "2021-05-21"),
}

new_entries = []
for report_id, (week, date_iso) in unrecoverable_report_ids.items():
    new_entries.append({
        "report_id": report_id,
        "report_title_original": f"An update of Lassa fever outbreak in Nigeria for Week {week}",
        "correction_type": "source_file_unrecoverable",
        "original_sitrep_week": week,
        "original_report_date_iso": date_iso,
        "corrected_sitrep_week": "",
        "corrected_report_date_iso": "",
        "justification": (
            "PDF page tree is structurally invalid ('root of pages tree has no "
            "/Kids array'). Confirmed on a fresh re-download (new SHA-256 hash) "
            "and unrecoverable via pikepdf/QPDF repair. Source file on NCDC's "
            "server is broken, not a download or parsing issue on our side. "
            "Excluded from automated extraction; case counts for this week are "
            "missing unless manually transcribed from a visual PDF viewer."
        ),
        "reviewer_initials": "",
        "review_date": now,
    })

all_corrections = existing_corrections + new_entries

with corrections_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_corrections)

print("Manual corrections log updated.")
print("Total corrections now logged:", len(all_corrections))
print()
print("Summary of all corrections:")
for c in all_corrections:
    print(f"  - {c['report_title_original']} ({c['original_report_date_iso']}): {c['correction_type']}")

In [ ]:
# Phase 4, Step 3: test weekly confirmed-case extraction on sample reports
# spanning the 2020-2025 training window, before running on the full corpus.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

# Exclude the confirmed Yellow Fever mislink from consideration
EXCLUDED_REPORT_IDS = {"f2c55a6e97a5e581"}

usable_2020_2025 = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
    and "2020" <= r["report_date_iso"][:4] <= "2025"
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
usable_2020_2025.sort(key=lambda r: r["report_date_iso"])

# Pick 8 samples spread evenly across the window for a first accuracy check
sample_count = 8
step = max(1, len(usable_2020_2025) // sample_count)
sample_rows = usable_2020_2025[::step][:sample_count]

# ---------------------------------------------------------------------------
# Extraction pattern: after "Current week (week NN)" (any spacing), the
# next tokens in order are Suspected, Confirmed, Probable, Deaths, CFR%.
# Text may have no spaces at all (compact) or normal spacing (spaced), and
# numbers may be split across whitespace/newlines, so we normalize first.
# ---------------------------------------------------------------------------

CURRENT_WEEK_PATTERN = re.compile(
    r"current\s*week\s*\(?\s*week\s*(\d{1,2})\s*\)?", re.IGNORECASE
)
NUMBER_TOKEN_PATTERN = re.compile(r"[\d,]+\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def extract_weekly_confirmed(text):
    """Find the Table 1 'current week' row and pull Suspected/Confirmed/
    Probable/Deaths/CFR as the first 5 numeric tokens after the label."""
    match = CURRENT_WEEK_PATTERN.search(text)
    if not match:
        return None

    tail = text[match.end():match.end() + 200]
    tokens = NUMBER_TOKEN_PATTERN.findall(tail)

    if len(tokens) < 5:
        return None

    def clean_int(tok):
        return int(tok.replace(",", "").replace("%", ""))

    return {
        "matched_week": match.group(1),
        "suspected_cases": clean_int(tokens[0]),
        "confirmed_cases": clean_int(tokens[1]),
        "probable_cases": clean_int(tokens[2]),
        "deaths": clean_int(tokens[3]),
        "cfr_percent": tokens[4].replace("%", ""),
        "raw_tail": tail,
    }

print(f"Testing extraction on {len(sample_rows)} sample reports:")
print()

for row in sample_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)
    result = extract_weekly_confirmed(text)

    print("=" * 80)
    print("Report:", row["report_title"], "|", row["report_date_iso"])
    print("Catalogue sitrep_week:", row["sitrep_week"])

    if result is None:
        print("EXTRACTION FAILED - no match found")
    else:
        print("Matched week in text:", result["matched_week"])
        print("Suspected cases:", result["suspected_cases"])
        print("Confirmed cases:", result["confirmed_cases"])
        print("Probable cases:", result["probable_cases"])
        print("Deaths:", result["deaths"])
        print("CFR%:", result["cfr_percent"])
        week_matches = result["matched_week"] == row["sitrep_week"] or \
                        result["matched_week"].lstrip("0") == row["sitrep_week"]
        print("Week number matches catalogue:", week_matches)
    print()

In [ ]:
# Phase 4, Step 3b: diagnose the Week 1, 2020 extraction failure
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

target_row = next(
    r for r in catalogue_rows
    if r["report_date_iso"] == "2020-01-04" and r["sitrep_week"] == "1"
)

local_path = Path(log_rows[target_row["report_id"]]["local_file"])
print("File:", local_path.name)
print()

doc = fitz.open(local_path)
text = "\n".join(page.get_text() for page in doc)
doc.close()

print("First 2500 characters of extracted text:")
print(text[:2500])

In [ ]:
# Phase 4, Step 3c: dual-template extraction (early-2020 vs standard format),
# tested across the first several weeks of 2020 specifically.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {"f2c55a6e97a5e581"}  # confirmed Yellow Fever mislink

early_2020_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
    and r["report_date_iso"][:7] in ("2020-01", "2020-02", "2020-03")
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
early_2020_rows.sort(key=lambda r: r["report_date_iso"])

# ---------------------------------------------------------------------------
# Standard-format pattern (2020-mid onward): "current week (week NN)" label,
# followed by Suspected, Confirmed, Probable, Deaths, CFR in that order.
# ---------------------------------------------------------------------------
STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(?\s*week\s*(\d{1,2})\s*\)?", re.IGNORECASE
)

# ---------------------------------------------------------------------------
# Early-2020 format: "Summary of current week indicators" label, followed
# by Suspected(Negative), Confirmed, Deaths, CFR - no Probable column.
# ---------------------------------------------------------------------------
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)

NUMBER_TOKEN_PATTERN = re.compile(r"[\d,]+\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    return int(tok.replace(",", "").replace("%", ""))

def extract_weekly_confirmed(text):
    """Try standard format first, then early-2020 format. Returns a dict
    with extraction_format noting which pattern matched, or None."""

    standard_match = STANDARD_LABEL.search(text)
    if standard_match:
        tail = text[standard_match.end():standard_match.end() + 200]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 5:
            return {
                "extraction_format": "standard_5field",
                "matched_week": standard_match.group(1),
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[1]),
                "probable_cases": clean_int(tokens[2]),
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", ""),
            }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        # Suspected cases may include a parenthetical "(Negative)" count
        # immediately after it, e.g. "98 (80) 18 2 11.1%" -> suspected=98,
        # negative=80, confirmed=18, deaths=2, cfr=11.1
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

print(f"Testing dual-template extraction on {len(early_2020_rows)} early-2020 reports:")
print()

for row in early_2020_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)
    result = extract_weekly_confirmed(text)

    print("=" * 80)
    print("Report:", row["report_title"], "|", row["report_date_iso"])
    print("Catalogue sitrep_week:", row["sitrep_week"])

    if result is None:
        print("EXTRACTION FAILED - no pattern matched")
    else:
        print("Format used:", result["extraction_format"])
        print("Matched week in text:", result["matched_week"])
        print("Suspected cases:", result["suspected_cases"])
        print("Confirmed cases:", result["confirmed_cases"])
        print("Probable cases:", result["probable_cases"])
        print("Deaths:", result["deaths"])
        print("CFR%:", result["cfr_percent"])
    print()

In [ ]:
# Phase 4, Step 3d: diagnose the weeks 4-6, 2020 extraction failures
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

target_dates = {"2020-01-25", "2020-02-01", "2020-02-08"}

failed_rows = [r for r in catalogue_rows if r["report_date_iso"] in target_dates]
failed_rows.sort(key=lambda r: r["report_date_iso"])

def extract_text(pdf_path, max_chars=2500):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

for row in failed_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    print("=" * 80)
    print("Report:", row["report_title"], "|", row["report_date_iso"])
    print("File:", local_path.name)
    print("=" * 80)
    text = extract_text(local_path)
    print(text)
    print()

In [ ]:
# Phase 4, Step 3f: fix the number-token regex (require a leading digit),
# then re-test across early-2020 reports.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {"f2c55a6e97a5e581"}

early_2020_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
    and r["report_date_iso"][:7] in ("2020-01", "2020-02", "2020-03")
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
early_2020_rows.sort(key=lambda r: r["report_date_iso"])

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(?\s*(?:week\s*)?(\d{1,2})\s*\)?\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)

# Fixed: must START with a digit, so a bare comma can never match alone.
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token after cleaning: {tok!r}")
    return int(float(cleaned))  # float() first in case a stray decimal slips through

def extract_weekly_confirmed(text):
    standard_match = STANDARD_LABEL.search(text)
    if standard_match:
        tail = text[standard_match.end():standard_match.end() + 200]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 5:
            return {
                "extraction_format": "standard_5field",
                "matched_week": standard_match.group(1),
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[1]),
                "probable_cases": clean_int(tokens[2]),
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", ""),
            }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

print(f"Re-testing on {len(early_2020_rows)} early-2020 reports:")
print()

fail_count = 0
error_count = 0
for row in early_2020_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        error_count += 1
        print(f"[ERROR] {row['report_title']} | {row['report_date_iso']} | {exc}")
        continue

    if result is None:
        fail_count += 1
        print(f"[FAILED] {row['report_title']} | {row['report_date_iso']}")
    else:
        week_ok = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")
        print(f"[OK] {row['report_title']} | {row['report_date_iso']} | "
              f"format={result['extraction_format']} | confirmed={result['confirmed_cases']} | "
              f"week_matches={week_ok}")

print()
print(f"Failures: {fail_count} | Errors: {error_count} | Total: {len(early_2020_rows)}")

In [ ]:
# Phase 4, Step 3g: diagnostic version that prints exactly what matched
# and why extraction failed for weeks 7-13, 2020.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

target_dates = {"2020-02-15", "2020-02-22"}  # just 2 samples to inspect closely

failed_rows = [r for r in catalogue_rows if r["report_date_iso"] in target_dates]

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(?\s*(?:week\s*)?(\d{1,2})\s*\)?\s*(?:indicators)?",
    re.IGNORECASE
)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

for row in failed_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)

    print("=" * 80)
    print("Report:", row["report_title"], "|", row["report_date_iso"])

    match = STANDARD_LABEL.search(text)
    if not match:
        print("NO LABEL MATCH AT ALL")
        print()
        continue

    print("Matched text:", repr(match.group(0)))
    print("Matched position:", match.start(), "-", match.end())
    print("Captured week group:", match.group(1))

    tail = text[match.end():match.end() + 200]
    print("Tail (200 chars after match):")
    print(repr(tail))

    tokens = NUMBER_TOKEN_PATTERN.findall(tail)
    print("Tokens found:", tokens)
    print("Token count:", len(tokens))
    print()

In [ ]:
# Phase 4, Step 3h: skip title-line matches, use the actual data-row
# occurrence of the "current week" label. Re-tested on the full early-2020 set.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {"f2c55a6e97a5e581"}

early_2020_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
    and r["report_date_iso"][:7] in ("2020-01", "2020-02", "2020-03")
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
early_2020_rows.sort(key=lambda r: r["report_date_iso"])

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(?\s*(?:week\s*)?(\d{1,2})\s*\)?\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token: {tok!r}")
    return int(float(cleaned))

def extract_weekly_confirmed(text):
    # Try every occurrence of the label, skipping title-line matches
    # (where the label is immediately followed by "cumulative", meaning
    # it's describing the table, not the actual data row).
    for match in STANDARD_LABEL.finditer(text):
        tail = text[match.end():match.end() + 250]
        tail_start = tail[:60].lower()
        if "cumulative" in tail_start:
            continue

        tokens = NUMBER_TOKEN_PATTERN.findall(tail[:200])
        if len(tokens) >= 5:
            return {
                "extraction_format": "standard_5field",
                "matched_week": match.group(1),
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[1]),
                "probable_cases": clean_int(tokens[2]),
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", ""),
            }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

print(f"Re-testing on {len(early_2020_rows)} early-2020 reports:")
print()

fail_count = 0
error_count = 0
for row in early_2020_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        error_count += 1
        print(f"[ERROR] {row['report_title']} | {row['report_date_iso']} | {exc}")
        continue

    if result is None:
        fail_count += 1
        print(f"[FAILED] {row['report_title']} | {row['report_date_iso']}")
    else:
        week_ok = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")
        print(f"[OK] {row['report_title']} | {row['report_date_iso']} | "
              f"format={result['extraction_format']} | confirmed={result['confirmed_cases']} | "
              f"week_matches={week_ok}")

print()
print(f"Failures: {fail_count} | Errors: {error_count} | Total: {len(early_2020_rows)}")

In [ ]:
# Phase 4, Step 3i: diagnose the Week 2, 2020 regression - why the
# standard-format pattern is now matching something it shouldn't.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

target_row = next(r for r in catalogue_rows if r["report_date_iso"] == "2020-01-11")
log_entry = log_rows[target_row["report_id"]]
local_path = Path(log_entry["local_file"])

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

text = extract_text(local_path)

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(?\s*(?:week\s*)?(\d{1,2})\s*\)?\s*(?:indicators)?",
    re.IGNORECASE
)

print("All STANDARD_LABEL matches in this document:")
print()
for i, match in enumerate(STANDARD_LABEL.finditer(text), start=1):
    print(f"--- Match {i} ---")
    print("Matched text:", repr(match.group(0)))
    print("Position:", match.start(), "-", match.end())
    tail = text[match.end():match.end() + 200]
    print("Tail:", repr(tail))
    print()

In [ ]:
# Phase 4, Step 3j: require parentheses around the week number (eliminates
# false matches like "current week 2020"), re-tested on all early-2020 reports.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {"f2c55a6e97a5e581"}

early_2020_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
    and r["report_date_iso"][:7] in ("2020-01", "2020-02", "2020-03")
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
early_2020_rows.sort(key=lambda r: r["report_date_iso"])

# Parentheses around the number are now MANDATORY, not optional. Every
# genuine Table 1 occurrence we've observed uses "(week NN)" or "(NN)" -
# only false matches (like "current week 2020") lack them.
STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token: {tok!r}")
    return int(float(cleaned))

def extract_weekly_confirmed(text):
    for match in STANDARD_LABEL.finditer(text):
        tail = text[match.end():match.end() + 250]
        if "cumulative" in tail[:60].lower():
            continue

        tokens = NUMBER_TOKEN_PATTERN.findall(tail[:200])
        if len(tokens) >= 5:
            return {
                "extraction_format": "standard_5field",
                "matched_week": match.group(1),
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[1]),
                "probable_cases": clean_int(tokens[2]),
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", ""),
            }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

print(f"Re-testing on {len(early_2020_rows)} early-2020 reports:")
print()

fail_count = 0
error_count = 0
for row in early_2020_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        error_count += 1
        print(f"[ERROR] {row['report_title']} | {row['report_date_iso']} | {exc}")
        continue

    if result is None:
        fail_count += 1
        print(f"[FAILED] {row['report_title']} | {row['report_date_iso']}")
    else:
        week_ok = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")
        print(f"[OK] {row['report_title']} | {row['report_date_iso']} | "
              f"format={result['extraction_format']} | confirmed={result['confirmed_cases']} | "
              f"week_matches={week_ok}")

print()
print(f"Failures: {fail_count} | Errors: {error_count} | Total: {len(early_2020_rows)}")

In [ ]:
# Phase 4, Step 4: run the validated dual-template extraction across the
# full 2020-2025 training-window corpus, with sanity checks and a saved
# weekly dataset ready for QA review.
# This cell is self-contained and imports everything it uses.

import re
import csv
import logging
from pathlib import Path
from datetime import datetime, timezone

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
corrections_csv_path = INTERIM_DIR / "manual_catalogue_corrections.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

with corrections_csv_path.open("r", encoding="utf-8") as f:
    corrections = {row["report_id"]: row for row in csv.DictReader(f)}

# Exclude: wrong-disease mislink, and the 3 structurally unrecoverable PDFs
EXCLUDED_REPORT_IDS = {
    r["report_id"] for r in corrections.values()
    if r["correction_type"] in ("exclude_wrong_disease", "source_file_unrecoverable")
}

training_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
    and "2020-01-01" <= r["report_date_iso"] <= "2025-12-31"
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
training_rows.sort(key=lambda r: r["report_date_iso"])

logger.info("Running extraction on %d reports in the 2020-2025 training window.", len(training_rows))

# ---------------------------------------------------------------------------
# Validated patterns from Step 3j
# ---------------------------------------------------------------------------

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token: {tok!r}")
    return int(float(cleaned))

def extract_weekly_confirmed(text):
    for match in STANDARD_LABEL.finditer(text):
        tail = text[match.end():match.end() + 250]
        if "cumulative" in tail[:60].lower():
            continue
        tokens = NUMBER_TOKEN_PATTERN.findall(tail[:200])
        if len(tokens) >= 5:
            return {
                "extraction_format": "standard_5field",
                "matched_week": match.group(1),
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[1]),
                "probable_cases": clean_int(tokens[2]),
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", ""),
            }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

# ---------------------------------------------------------------------------
# Run extraction with sanity checks
# ---------------------------------------------------------------------------

results = []
extraction_failures = []
extraction_errors = []
sanity_flags = []

for i, row in enumerate(training_rows, start=1):
    report_id = row["report_id"]
    local_path = Path(log_rows[report_id]["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        extraction_errors.append({
            "report_id": report_id, "report_title": row["report_title"],
            "report_date_iso": row["report_date_iso"], "error": str(exc),
        })
        continue

    if result is None:
        extraction_failures.append({
            "report_id": report_id, "report_title": row["report_title"],
            "report_date_iso": row["report_date_iso"],
        })
        continue

    week_matches = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")

    entry = {
        "report_id": report_id,
        "report_title": row["report_title"],
        "report_date_iso": row["report_date_iso"],
        "sitrep_week": row["sitrep_week"],
        "extraction_format": result["extraction_format"],
        "matched_week_in_text": result["matched_week"],
        "week_matches_catalogue": week_matches,
        "suspected_cases": result["suspected_cases"],
        "confirmed_cases": result["confirmed_cases"],
        "probable_cases": result["probable_cases"],
        "deaths": result["deaths"],
        "cfr_percent": result["cfr_percent"],
        "extracted_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    results.append(entry)

    # Sanity checks: flag implausible values for manual QA review
    flags = []
    if not week_matches:
        flags.append("week_mismatch")
    if result["confirmed_cases"] == 0:
        flags.append("zero_confirmed")
    if result["confirmed_cases"] > 300:
        flags.append("implausibly_large_confirmed")
    if result["probable_cases"] > result["suspected_cases"] and result["suspected_cases"] > 0:
        flags.append("probable_exceeds_suspected")
    if result["deaths"] > result["confirmed_cases"]:
        flags.append("deaths_exceed_confirmed")
    if result["confirmed_cases"] > result["suspected_cases"] and result["suspected_cases"] > 0:
        flags.append("confirmed_exceeds_suspected")

    if flags:
        sanity_flags.append({**entry, "flags": ";".join(flags)})

    if i % 100 == 0:
        logger.info("Extracted %d / %d reports so far.", i, len(training_rows))

# ---------------------------------------------------------------------------
# Save all outputs
# ---------------------------------------------------------------------------

weekly_csv_path = INTERIM_DIR / "weekly_extracted_raw.csv"
with weekly_csv_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(results[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

if sanity_flags:
    flags_csv_path = INTERIM_DIR / "weekly_extraction_sanity_flags.csv"
    with flags_csv_path.open("w", newline="", encoding="utf-8") as f:
        fieldnames = list(sanity_flags[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(sanity_flags)

if extraction_failures:
    failures_csv_path = INTERIM_DIR / "weekly_extraction_failures.csv"
    with failures_csv_path.open("w", newline="", encoding="utf-8") as f:
        fieldnames = list(extraction_failures[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(extraction_failures)

logger.info("Extraction complete. Success: %d, Failed: %d, Errors: %d, Flagged: %d",
            len(results), len(extraction_failures), len(extraction_errors), len(sanity_flags))

print("=" * 60)
print("EXTRACTION SUMMARY")
print("=" * 60)
print(f"Total reports in training window: {len(training_rows)}")
print(f"Successfully extracted: {len(results)}")
print(f"Extraction failures (no pattern matched): {len(extraction_failures)}")
print(f"Extraction errors (exception raised): {len(extraction_errors)}")
print(f"Flagged for sanity-check review: {len(sanity_flags)}")
print()
print("Weekly dataset saved to:", weekly_csv_path.resolve())

if extraction_failures:
    print()
    print("Failed reports:")
    for f in extraction_failures:
        print(f"  - {f['report_title']} | {f['report_date_iso']}")

if extraction_errors:
    print()
    print("Errored reports:")
    for e in extraction_errors:
        print(f"  - {e['report_title']} | {e['report_date_iso']} | {e['error']}")

if sanity_flags:
    print()
    print("Sample flagged rows (up to 10):")
    for f in sanity_flags[:10]:
        print(f"  - {f['report_title']} | {f['report_date_iso']} | flags: {f['flags']} | "
              f"confirmed={f['confirmed_cases']}, suspected={f['suspected_cases']}")

In [ ]:
# Phase 4, Step 4b: diagnose the 92 extraction failures by sampling across
# different years to spot whether it's one recurring pattern or several.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
failures_csv_path = INTERIM_DIR / "weekly_extraction_failures.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_by_id = {r["report_id"]: r for r in csv.DictReader(f)}

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

with failures_csv_path.open("r", encoding="utf-8") as f:
    failures = list(csv.DictReader(f))

# Sample one failure from each of several different years/eras
target_dates = {
    "2020-06-06",  # 2020, mid-year
    "2021-05-07",  # 2021
    "2022-01-08",  # 2022, start of year
    "2023-01-05",  # 2023
    "2024-01-11",  # 2024
}

samples = [f for f in failures if f["report_date_iso"] in target_dates]

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)

def extract_text(pdf_path, max_chars=3000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

for f in samples:
    report_id = f["report_id"]
    local_path = Path(log_rows[report_id]["local_file"])
    text = extract_text(local_path)

    print("=" * 80)
    print("Report:", f["report_title"], "|", f["report_date_iso"])

    # Does the standard pattern find ANY match at all in this doc?
    matches = list(STANDARD_LABEL.finditer(text))
    print(f"STANDARD_LABEL matches found: {len(matches)}")

    # Search for "current week" loosely to see how it's actually phrased
    loose_matches = [m.group(0) for m in re.finditer(r".{0,20}current\s*week.{0,30}", text, re.IGNORECASE)]
    print("Loose 'current week' contexts found:", loose_matches[:5])
    print()
    print("First 1200 characters of document:")
    print(text[:1200])
    print()

In [ ]:
# Phase 4, Step 4c: inspect both STANDARD_LABEL match positions and their
# tails directly, for one representative failing report per era.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
failures_csv_path = INTERIM_DIR / "weekly_extraction_failures.csv"

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

with failures_csv_path.open("r", encoding="utf-8") as f:
    failures = list(csv.DictReader(f))

target_dates = {"2020-06-06", "2022-01-08", "2023-01-05", "2024-01-11"}
samples = [f for f in failures if f["report_date_iso"] in target_dates]

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

for f in samples:
    report_id = f["report_id"]
    local_path = Path(log_rows[report_id]["local_file"])
    text = extract_text(local_path)

    print("=" * 80)
    print("Report:", f["report_title"], "|", f["report_date_iso"])

    for i, match in enumerate(STANDARD_LABEL.finditer(text), start=1):
        tail = text[match.end():match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail[:200])
        print(f"--- Match {i} at position {match.start()}-{match.end()} ---")
        print("Matched text:", repr(match.group(0)))
        print("Tail:", repr(tail[:250]))
        print("Tokens in tail:", tokens, "| count:", len(tokens))
        print()
    print()

In [ ]:
# Phase 4, Step 4d: replace the fragile "cumulative in first 60 chars" skip
# with a structural check (does the tail start with a digit?). Re-tested
# across both the previously-passing early-2020 batch and today's new
# failures, so we don't trade one regression for another.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {"f2c55a6e97a5e581"}

# Combined test set: the 13 early-2020 reports we already validated,
# plus the newly diagnosed failures spanning 2020-2024.
test_dates = {
    "2020-01-04", "2020-01-11", "2020-01-18", "2020-01-25", "2020-02-01",
    "2020-02-08", "2020-02-15", "2020-02-22", "2020-02-29", "2020-03-07",
    "2020-03-14", "2020-03-21", "2020-03-28",
    "2020-06-06", "2022-01-08", "2023-01-05", "2024-01-11",
}

test_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"] in test_dates
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
test_rows.sort(key=lambda r: r["report_date_iso"])

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")
LEADING_DIGIT_CHECK = re.compile(r"^\s*\d")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token: {tok!r}")
    return int(float(cleaned))

def extract_weekly_confirmed(text):
    for match in STANDARD_LABEL.finditer(text):
        tail = text[match.end():match.end() + 250]
        # A genuine data row's tail starts directly with a number.
        # A title-line match's tail starts with punctuation/text
        # ("cumulative from Epi week...").
        if not LEADING_DIGIT_CHECK.match(tail):
            continue
        tokens = NUMBER_TOKEN_PATTERN.findall(tail[:200])
        if len(tokens) >= 5:
            return {
                "extraction_format": "standard_5field",
                "matched_week": match.group(1),
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[1]),
                "probable_cases": clean_int(tokens[2]),
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", ""),
            }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

print(f"Testing on {len(test_rows)} reports (13 previously-passing + 4 newly-diagnosed):")
print()

fail_count = 0
error_count = 0
for row in test_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        error_count += 1
        print(f"[ERROR] {row['report_title']} | {row['report_date_iso']} | {exc}")
        continue

    if result is None:
        fail_count += 1
        print(f"[FAILED] {row['report_title']} | {row['report_date_iso']}")
    else:
        week_ok = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")
        print(f"[OK] {row['report_title']} | {row['report_date_iso']} | "
              f"format={result['extraction_format']} | confirmed={result['confirmed_cases']} | "
              f"week_matches={week_ok}")

print()
print(f"Failures: {fail_count} | Errors: {error_count} | Total: {len(test_rows)}")

In [ ]:
# Phase 4, Step 4e: correct check - scan for 'cumulative' only within the
# span actually consumed by the first 5 numeric tokens, not a fixed
# character window. Re-tested on the full 17-report combined set.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {"f2c55a6e97a5e581"}

test_dates = {
    "2020-01-04", "2020-01-11", "2020-01-18", "2020-01-25", "2020-02-01",
    "2020-02-08", "2020-02-15", "2020-02-22", "2020-02-29", "2020-03-07",
    "2020-03-14", "2020-03-21", "2020-03-28",
    "2020-06-06", "2022-01-08", "2023-01-05", "2024-01-11",
}

test_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"] in test_dates
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
test_rows.sort(key=lambda r: r["report_date_iso"])

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token: {tok!r}")
    return int(float(cleaned))

def extract_weekly_confirmed(text):
    for match in STANDARD_LABEL.finditer(text):
        tail = text[match.end():match.end() + 300]
        token_matches = list(NUMBER_TOKEN_PATTERN.finditer(tail))

        if len(token_matches) < 5:
            continue

        # The span actually consumed to reach the 5th number - if the
        # word "cumulative" appears anywhere in that span, this is a
        # title-line false match (numbers embedded in "cumulative from
        # Epi week 01-23, 2020..."), not a genuine data row.
        span_to_fifth_token = tail[:token_matches[4].end()]
        if "cumulative" in span_to_fifth_token.lower():
            continue

        tokens = [m.group(0) for m in token_matches[:5]]
        return {
            "extraction_format": "standard_5field",
            "matched_week": match.group(1),
            "suspected_cases": clean_int(tokens[0]),
            "confirmed_cases": clean_int(tokens[1]),
            "probable_cases": clean_int(tokens[2]),
            "deaths": clean_int(tokens[3]),
            "cfr_percent": tokens[4].replace("%", ""),
        }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

print(f"Testing on {len(test_rows)} reports:")
print()

fail_count = 0
error_count = 0
for row in test_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        error_count += 1
        print(f"[ERROR] {row['report_title']} | {row['report_date_iso']} | {exc}")
        continue

    if result is None:
        fail_count += 1
        print(f"[FAILED] {row['report_title']} | {row['report_date_iso']}")
    else:
        week_ok = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")
        print(f"[OK] {row['report_title']} | {row['report_date_iso']} | "
              f"format={result['extraction_format']} | confirmed={result['confirmed_cases']} | "
              f"week_matches={week_ok}")

print()
print(f"Failures: {fail_count} | Errors: {error_count} | Total: {len(test_rows)}")

In [ ]:
# Phase 4, Step 4e: correct check - scan for 'cumulative' only within the
# span actually consumed by the first 5 numeric tokens, not a fixed
# character window. Re-tested on the full 17-report combined set.
# This cell is self-contained and imports everything it uses.

import re
import csv
from pathlib import Path

import fitz  # PyMuPDF

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {"f2c55a6e97a5e581"}

test_dates = {
    "2020-01-04", "2020-01-11", "2020-01-18", "2020-01-25", "2020-02-01",
    "2020-02-08", "2020-02-15", "2020-02-22", "2020-02-29", "2020-03-07",
    "2020-03-14", "2020-03-21", "2020-03-28",
    "2020-06-06", "2022-01-08", "2023-01-05", "2024-01-11",
}

test_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"] in test_dates
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
test_rows.sort(key=lambda r: r["report_date_iso"])

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = fitz.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token: {tok!r}")
    return int(float(cleaned))

def extract_weekly_confirmed(text):
    for match in STANDARD_LABEL.finditer(text):
        tail = text[match.end():match.end() + 300]
        token_matches = list(NUMBER_TOKEN_PATTERN.finditer(tail))

        if len(token_matches) < 5:
            continue

        # The span actually consumed to reach the 5th number - if the
        # word "cumulative" appears anywhere in that span, this is a
        # title-line false match (numbers embedded in "cumulative from
        # Epi week 01-23, 2020..."), not a genuine data row.
        span_to_fifth_token = tail[:token_matches[4].end()]
        if "cumulative" in span_to_fifth_token.lower():
            continue

        tokens = [m.group(0) for m in token_matches[:5]]
        return {
            "extraction_format": "standard_5field",
            "matched_week": match.group(1),
            "suspected_cases": clean_int(tokens[0]),
            "confirmed_cases": clean_int(tokens[1]),
            "probable_cases": clean_int(tokens[2]),
            "deaths": clean_int(tokens[3]),
            "cfr_percent": tokens[4].replace("%", ""),
        }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

print(f"Testing on {len(test_rows)} reports:")
print()

fail_count = 0
error_count = 0
for row in test_rows:
    log_entry = log_rows[row["report_id"]]
    local_path = Path(log_entry["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        error_count += 1
        print(f"[ERROR] {row['report_title']} | {row['report_date_iso']} | {exc}")
        continue

    if result is None:
        fail_count += 1
        print(f"[FAILED] {row['report_title']} | {row['report_date_iso']}")
    else:
        week_ok = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")
        print(f"[OK] {row['report_title']} | {row['report_date_iso']} | "
              f"format={result['extraction_format']} | confirmed={result['confirmed_cases']} | "
              f"week_matches={week_ok}")

print()
print(f"Failures: {fail_count} | Errors: {error_count} | Total: {len(test_rows)}")

In [ ]:
# Phase 4, Step 5: run the corrected dual-template extraction across the
# full 2020-2025 training-window corpus (using the validated span-based
# 'cumulative' check), with sanity checks.
# This cell is self-contained and imports everything it uses.

import re
import csv
import logging
from pathlib import Path
from datetime import datetime, timezone

import pymupdf

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
LOGS_DIR = PROJECT_ROOT / "logs"

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
corrections_csv_path = INTERIM_DIR / "manual_catalogue_corrections.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

with corrections_csv_path.open("r", encoding="utf-8") as f:
    corrections = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {
    r["report_id"] for r in corrections.values()
    if r["correction_type"] in ("exclude_wrong_disease", "source_file_unrecoverable")
}

training_rows = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
    and "2020-01-01" <= r["report_date_iso"] <= "2025-12-31"
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
training_rows.sort(key=lambda r: r["report_date_iso"])

logger.info("Running extraction on %d reports in the 2020-2025 training window.", len(training_rows))

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = pymupdf.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token: {tok!r}")
    return int(float(cleaned))

def extract_weekly_confirmed(text):
    for match in STANDARD_LABEL.finditer(text):
        tail = text[match.end():match.end() + 300]
        token_matches = list(NUMBER_TOKEN_PATTERN.finditer(tail))

        if len(token_matches) < 5:
            continue

        span_to_fifth_token = tail[:token_matches[4].end()]
        if "cumulative" in span_to_fifth_token.lower():
            continue

        tokens = [m.group(0) for m in token_matches[:5]]
        return {
            "extraction_format": "standard_5field",
            "matched_week": match.group(1),
            "suspected_cases": clean_int(tokens[0]),
            "confirmed_cases": clean_int(tokens[1]),
            "probable_cases": clean_int(tokens[2]),
            "deaths": clean_int(tokens[3]),
            "cfr_percent": tokens[4].replace("%", ""),
        }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

results = []
extraction_failures = []
extraction_errors = []
sanity_flags = []

for i, row in enumerate(training_rows, start=1):
    report_id = row["report_id"]
    local_path = Path(log_rows[report_id]["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        extraction_errors.append({
            "report_id": report_id, "report_title": row["report_title"],
            "report_date_iso": row["report_date_iso"], "error": str(exc),
        })
        continue

    if result is None:
        extraction_failures.append({
            "report_id": report_id, "report_title": row["report_title"],
            "report_date_iso": row["report_date_iso"],
        })
        continue

    week_matches = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")

    entry = {
        "report_id": report_id,
        "report_title": row["report_title"],
        "report_date_iso": row["report_date_iso"],
        "sitrep_week": row["sitrep_week"],
        "extraction_format": result["extraction_format"],
        "matched_week_in_text": result["matched_week"],
        "week_matches_catalogue": week_matches,
        "suspected_cases": result["suspected_cases"],
        "confirmed_cases": result["confirmed_cases"],
        "probable_cases": result["probable_cases"],
        "deaths": result["deaths"],
        "cfr_percent": result["cfr_percent"],
        "extracted_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    results.append(entry)

    flags = []
    if not week_matches:
        flags.append("week_mismatch")
    if result["confirmed_cases"] == 0:
        flags.append("zero_confirmed")
    if result["confirmed_cases"] > 300:
        flags.append("implausibly_large_confirmed")
    if result["probable_cases"] > result["suspected_cases"] and result["suspected_cases"] > 0:
        flags.append("probable_exceeds_suspected")
    if result["deaths"] > result["confirmed_cases"]:
        flags.append("deaths_exceed_confirmed")
    if result["confirmed_cases"] > result["suspected_cases"] and result["suspected_cases"] > 0:
        flags.append("confirmed_exceeds_suspected")

    if flags:
        sanity_flags.append({**entry, "flags": ";".join(flags)})

    if i % 100 == 0:
        logger.info("Extracted %d / %d reports so far.", i, len(training_rows))

weekly_csv_path = INTERIM_DIR / "weekly_extracted_raw.csv"
with weekly_csv_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(results[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

if sanity_flags:
    flags_csv_path = INTERIM_DIR / "weekly_extraction_sanity_flags.csv"
    with flags_csv_path.open("w", newline="", encoding="utf-8") as f:
        fieldnames = list(sanity_flags[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(sanity_flags)

if extraction_failures:
    failures_csv_path = INTERIM_DIR / "weekly_extraction_failures.csv"
    with failures_csv_path.open("w", newline="", encoding="utf-8") as f:
        fieldnames = list(extraction_failures[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(extraction_failures)

logger.info("Extraction complete. Success: %d, Failed: %d, Errors: %d, Flagged: %d",
            len(results), len(extraction_failures), len(extraction_errors), len(sanity_flags))

print("=" * 60)
print("EXTRACTION SUMMARY")
print("=" * 60)
print(f"Total reports in training window: {len(training_rows)}")
print(f"Successfully extracted: {len(results)}")
print(f"Extraction failures (no pattern matched): {len(extraction_failures)}")
print(f"Extraction errors (exception raised): {len(extraction_errors)}")
print(f"Flagged for sanity-check review: {len(sanity_flags)}")
print()
print("Weekly dataset saved to:", weekly_csv_path.resolve())

if extraction_failures:
    print()
    print("Failed reports:")
    for f in extraction_failures:
        print(f"  - {f['report_title']} | {f['report_date_iso']}")

if extraction_errors:
    print()
    print("Errored reports:")
    for e in extraction_errors:
        print(f"  - {e['report_title']} | {e['report_date_iso']} | {e['error']}")

if sanity_flags:
    print()
    print("Sample flagged rows (up to 15):")
    for f in sanity_flags[:15]:
        print(f"  - {f['report_title']} | {f['report_date_iso']} | flags: {f['flags']} | "
              f"confirmed={f['confirmed_cases']}, suspected={f['suspected_cases']}")

In [ ]:
# Phase 4, Step 6: build the QA file marking the 7 unresolved reports as
# pending_review, and produce the approved weekly dataset (excluding them)
# ready for monthly aggregation.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

weekly_csv_path = INTERIM_DIR / "weekly_extracted_raw.csv"
failures_csv_path = INTERIM_DIR / "weekly_extraction_failures.csv"
flags_csv_path = INTERIM_DIR / "weekly_extraction_sanity_flags.csv"

with weekly_csv_path.open("r", encoding="utf-8") as f:
    weekly_rows = list(csv.DictReader(f))

with failures_csv_path.open("r", encoding="utf-8") as f:
    failure_rows = list(csv.DictReader(f))

with flags_csv_path.open("r", encoding="utf-8") as f:
    flag_rows = list(csv.DictReader(f))

flagged_report_ids = {r["report_id"] for r in flag_rows}
now = datetime.now(timezone.utc).isoformat()

# ---------------------------------------------------------------------------
# Build the QA workbook per the original Phase 4 spec: analyst_decision,
# reviewed_new_confirmed_cases, reviewer_initials, review_date, review_notes.
# ---------------------------------------------------------------------------

qa_rows = []

# Extraction failures: no data at all, need manual review
for f in failure_rows:
    qa_rows.append({
        "report_id": f["report_id"],
        "report_title": f["report_title"],
        "report_date_iso": f["report_date_iso"],
        "extraction_status": "extraction_failed",
        "auto_confirmed_cases": "",
        "flags": "no_pattern_matched",
        "analyst_decision": "pending_review",
        "reviewed_new_confirmed_cases": "",
        "reviewer_initials": "",
        "review_date": "",
        "review_notes": "",
        "audit_sample": "N",
    })

# Sanity-flagged rows: data was extracted but looks suspicious
for entry in weekly_rows:
    if entry["report_id"] in flagged_report_ids:
        flag_info = next(f for f in flag_rows if f["report_id"] == entry["report_id"])
        qa_rows.append({
            "report_id": entry["report_id"],
            "report_title": entry["report_title"],
            "report_date_iso": entry["report_date_iso"],
            "extraction_status": "extracted_flagged",
            "auto_confirmed_cases": entry["confirmed_cases"],
            "flags": flag_info["flags"],
            "analyst_decision": "pending_review",
            "reviewed_new_confirmed_cases": "",
            "reviewer_initials": "",
            "review_date": "",
            "review_notes": "",
            "audit_sample": "N",
        })

qa_csv_path = INTERIM_DIR / "weekly_qa_workbook.csv"
with qa_csv_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(qa_rows[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(qa_rows)

# ---------------------------------------------------------------------------
# Approved weekly dataset: everything extracted successfully MINUS anything
# flagged. This is what monthly aggregation will use.
# ---------------------------------------------------------------------------

pending_report_ids = flagged_report_ids  # failures were never in weekly_rows anyway

approved_rows = [
    r for r in weekly_rows
    if r["report_id"] not in pending_report_ids
]

approved_csv_path = PROCESSED_DIR / "weekly_approved.csv"
with approved_csv_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(approved_rows[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(approved_rows)

print("QA workbook saved to:", qa_csv_path.resolve())
print(f"  Total rows needing review: {len(qa_rows)}")
print(f"    Extraction failures: {len(failure_rows)}")
print(f"    Sanity-flagged: {len(flag_rows)}")
print()
print("Approved weekly dataset saved to:", approved_csv_path.resolve())
print(f"  Approved rows (ready for aggregation): {len(approved_rows)}")
print(f"  Excluded (pending review): {len(weekly_rows) - len(approved_rows)}")

In [ ]:
# Phase 5, Step 1 (fixed): aggregate approved weekly data to monthly totals,
# with a separate, correctly-scoped file for boundary-sensitive weeks.
# This cell is self-contained and imports everything it uses.

import csv
import calendar
from pathlib import Path
from datetime import date
from collections import defaultdict

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

approved_csv_path = PROCESSED_DIR / "weekly_approved.csv"
with approved_csv_path.open("r", encoding="utf-8") as f:
    approved_rows = list(csv.DictReader(f))

def days_from_month_boundary(d):
    last_day_of_month = calendar.monthrange(d.year, d.month)[1]
    return min(d.day - 1, last_day_of_month - d.day)

monthly_totals = defaultdict(lambda: {
    "confirmed_cases": 0, "suspected_cases": 0, "deaths": 0,
    "week_count": 0, "report_ids": []
})

boundary_sensitive_weeks = []

for row in approved_rows:
    d = date.fromisoformat(row["report_date_iso"])
    month_key = f"{d.year:04d}-{d.month:02d}"

    monthly_totals[month_key]["confirmed_cases"] += int(row["confirmed_cases"])
    monthly_totals[month_key]["suspected_cases"] += int(row["suspected_cases"])
    monthly_totals[month_key]["deaths"] += int(row["deaths"])
    monthly_totals[month_key]["week_count"] += 1
    monthly_totals[month_key]["report_ids"].append(row["report_id"])

    if days_from_month_boundary(d) <= 2:
        boundary_sensitive_weeks.append({
            "report_id": row["report_id"],
            "report_title": row["report_title"],
            "report_date_iso": row["report_date_iso"],
            "assigned_month": month_key,
            "days_from_boundary": days_from_month_boundary(d),
        })

expected_months = [f"{year:04d}-{month:02d}" for year in range(2020, 2026) for month in range(1, 13)]

monthly_rows = []
missing_months = []

for month_key in expected_months:
    if month_key not in monthly_totals:
        missing_months.append(month_key)
        monthly_rows.append({
            "year_month": month_key,
            "confirmed_cases": "",
            "suspected_cases": "",
            "deaths": "",
            "week_count": 0,
            "status": "MISSING",
        })
    else:
        data = monthly_totals[month_key]
        status = "OK" if 3 <= data["week_count"] <= 5 else "UNUSUAL_WEEK_COUNT"
        monthly_rows.append({
            "year_month": month_key,
            "confirmed_cases": data["confirmed_cases"],
            "suspected_cases": data["suspected_cases"],
            "deaths": data["deaths"],
            "week_count": data["week_count"],
            "status": status,
        })

# --- Write monthly dataset (its own fieldnames variable) ---
monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
monthly_fieldnames = list(monthly_rows[0].keys())
with monthly_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=monthly_fieldnames)
    writer.writeheader()
    writer.writerows(monthly_rows)

# --- Write boundary-sensitive weeks (its own separate fieldnames variable) ---
if boundary_sensitive_weeks:
    boundary_csv_path = PROCESSED_DIR / "month_boundary_sensitive_weeks.csv"
    boundary_fieldnames = list(boundary_sensitive_weeks[0].keys())
    with boundary_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=boundary_fieldnames)
        writer.writeheader()
        writer.writerows(boundary_sensitive_weeks)

print("=" * 60)
print("MONTHLY AGGREGATION SUMMARY")
print("=" * 60)
print(f"Expected months (Jan 2020 - Dec 2025): {len(expected_months)}")
print(f"Months with data: {len(expected_months) - len(missing_months)}")
print(f"Missing months: {len(missing_months)}")
if missing_months:
    print("  Missing:", missing_months)
print()
unusual = [r for r in monthly_rows if r["status"] == "UNUSUAL_WEEK_COUNT"]
print(f"Months with unusual week counts (not 3-5 weeks): {len(unusual)}")
for r in unusual:
    print(f"  - {r['year_month']}: {r['week_count']} weeks")
print()
print(f"Weeks within 2 days of a month boundary (assumption-sensitive): {len(boundary_sensitive_weeks)}")
print()
print("Monthly dataset saved to:", monthly_csv_path.resolve())
print()
print("First 12 months preview:")
for r in monthly_rows[:12]:
    print(f"  {r['year_month']}: confirmed={r['confirmed_cases']}, weeks={r['week_count']}, status={r['status']}")

In [ ]:
# Phase 5, Step 2: diagnose the 3 under-covered months - are the missing
# weeks the ones we excluded as pending_review, or are they absent from
# the catalogue entirely?
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path
from datetime import date

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
corrections_csv_path = INTERIM_DIR / "manual_catalogue_corrections.csv"
qa_csv_path = INTERIM_DIR / "weekly_qa_workbook.csv"
approved_csv_path = PROCESSED_DIR / "weekly_approved.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with corrections_csv_path.open("r", encoding="utf-8") as f:
    corrections = {row["report_id"]: row for row in csv.DictReader(f)}

with qa_csv_path.open("r", encoding="utf-8") as f:
    qa_rows = {row["report_id"]: row for row in csv.DictReader(f)}

with approved_csv_path.open("r", encoding="utf-8") as f:
    approved_report_ids = {row["report_id"] for row in csv.DictReader(f)}

target_months = {"2021-06", "2021-08", "2022-05"}

for month_key in sorted(target_months):
    print("=" * 70)
    print(f"Month: {month_key}")
    print("=" * 70)

    month_rows = [
        r for r in catalogue_rows
        if r["report_date_iso"] and r["report_date_iso"][:7] == month_key
    ]
    month_rows.sort(key=lambda r: r["report_date_iso"])

    if not month_rows:
        print("  No catalogue entries at all for this month - genuine NCDC gap or date-labeling issue.")
        continue

    for r in month_rows:
        report_id = r["report_id"]
        if report_id in approved_report_ids:
            status = "APPROVED (counted in monthly total)"
        elif report_id in qa_rows:
            status = f"PENDING_REVIEW ({qa_rows[report_id]['extraction_status']}, flags: {qa_rows[report_id]['flags']})"
        elif report_id in corrections:
            status = f"EXCLUDED ({corrections[report_id]['correction_type']})"
        else:
            status = "UNKNOWN STATUS - not in approved, QA, or corrections"

        print(f"  - {r['report_title']} | {r['report_date_iso']} | {status}")
    print()

In [ ]:
# Phase 5, Step 3: trace the two "UNKNOWN STATUS" reports back through
# every stage of the pipeline to find where they disappeared.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
weekly_raw_csv_path = INTERIM_DIR / "weekly_extracted_raw.csv"
failures_csv_path = INTERIM_DIR / "weekly_extraction_failures.csv"
flags_csv_path = INTERIM_DIR / "weekly_extraction_sanity_flags.csv"
corrections_csv_path = INTERIM_DIR / "manual_catalogue_corrections.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_by_id = {r["report_id"]: r for r in csv.DictReader(f)}

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_by_id = {r["report_id"]: r for r in csv.DictReader(f)}

with weekly_raw_csv_path.open("r", encoding="utf-8") as f:
    weekly_raw_ids = {r["report_id"] for r in csv.DictReader(f)}

with failures_csv_path.open("r", encoding="utf-8") as f:
    failure_ids = {r["report_id"] for r in csv.DictReader(f)}

with flags_csv_path.open("r", encoding="utf-8") as f:
    flag_ids = {r["report_id"] for r in csv.DictReader(f)}

with corrections_csv_path.open("r", encoding="utf-8") as f:
    correction_ids = {r["report_id"] for r in csv.DictReader(f)}

target_dates = {"2021-06-11", "2022-05-19"}

for r in catalogue_by_id.values():
    if r["report_date_iso"] not in target_dates:
        continue

    report_id = r["report_id"]
    print("=" * 70)
    print("Report:", r["report_title"], "|", r["report_date_iso"])
    print("Report ID:", report_id)
    print()
    print("In catalogue:", True)

    log_entry = log_by_id.get(report_id)
    if log_entry is None:
        print("In acquisition log: NOT FOUND")
    else:
        print("In acquisition log: YES")
        print("  download_status:", log_entry.get("download_status"))
        print("  error_message:", log_entry.get("error_message"))
        print("  local_file:", log_entry.get("local_file"))
        if log_entry.get("local_file"):
            exists = Path(log_entry["local_file"]).exists()
            print("  file exists on disk:", exists)

    print("In weekly_extracted_raw.csv (successful extraction):", report_id in weekly_raw_ids)
    print("In extraction failures list:", report_id in failure_ids)
    print("In sanity flags list:", report_id in flag_ids)
    print("In manual corrections/exclusions list:", report_id in correction_ids)
    print()

In [ ]:
# Phase 5, Step 4b: document the 2 unrecovered downloads as source-unavailable
# in the corrections log, so the gap is explicit and auditable rather than silent.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
corrections_csv_path = INTERIM_DIR / "manual_catalogue_corrections.csv"

with corrections_csv_path.open("r", encoding="utf-8") as f:
    existing_corrections = list(csv.DictReader(f))
    fieldnames = list(existing_corrections[0].keys())

now = datetime.now(timezone.utc).isoformat()

skipped_reports = {
    "7906c9d07aae22d8": ("24", "2021-06-11"),
    "6f408311e91c866f": ("21", "2022-05-19"),
}

new_entries = []
for report_id, (week, date_iso) in skipped_reports.items():
    new_entries.append({
        "report_id": report_id,
        "report_title_original": f"An update of Lassa fever outbreak in Nigeria for Week {week}",
        "correction_type": "source_file_unrecoverable",
        "original_sitrep_week": week,
        "original_report_date_iso": date_iso,
        "corrected_sitrep_week": "",
        "corrected_report_date_iso": "",
        "justification": (
            "Original bulk download (Phase 3) returned HTTP 404 for this report. "
            "Decision made to accept as a documented data gap rather than retry, "
            "given the small number of affected weeks (2 of 313) relative to the "
            "full training window. Excluded from monthly aggregation."
        ),
        "reviewer_initials": "",
        "review_date": now,
    })

all_corrections = existing_corrections + new_entries

with corrections_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_corrections)

print("Corrections log updated.")
print("Total corrections now logged:", len(all_corrections))
print()
print("Full list of known data gaps in the 2020-2025 training window:")
for c in all_corrections:
    print(f"  - {c['report_title_original']} ({c['original_report_date_iso']}): {c['correction_type']}")

In [ ]:
# Phase 5.5: Descriptive analysis of the Lassa fever surveillance data
# before any forecasting. This cell is self-contained and imports
# everything it uses.

import csv
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
weekly_csv_path = PROCESSED_DIR / "weekly_approved.csv"

monthly_df = pd.read_csv(monthly_csv_path)
monthly_df["confirmed_cases"] = pd.to_numeric(monthly_df["confirmed_cases"], errors="coerce")
monthly_df["date"] = pd.to_datetime(monthly_df["year_month"], format="%Y-%m")
monthly_df["year"] = monthly_df["date"].dt.year
monthly_df["month"] = monthly_df["date"].dt.month
monthly_df["month_name"] = monthly_df["date"].dt.strftime("%b")
monthly_df = monthly_df.sort_values("date").reset_index(drop=True)

weekly_df = pd.read_csv(weekly_csv_path)
weekly_df["report_date_iso"] = pd.to_datetime(weekly_df["report_date_iso"])
weekly_df["confirmed_cases"] = pd.to_numeric(weekly_df["confirmed_cases"])
weekly_df["deaths"] = pd.to_numeric(weekly_df["deaths"])
weekly_df["suspected_cases"] = pd.to_numeric(weekly_df["suspected_cases"])

In [ ]:
# ---------------------------------------------------------------------------
# 1. Overall summary statistics (monthly series, 2020-2025 only)
# ---------------------------------------------------------------------------

analysis_df = monthly_df[(monthly_df["date"] >= "2020-01-01") & (monthly_df["date"] <= "2025-12-31")].copy()

print("=" * 70)
print("1. OVERALL MONTHLY SUMMARY STATISTICS (Jan 2020 - Dec 2025)")
print("=" * 70)
print(analysis_df["confirmed_cases"].describe().round(1))
print()
print(f"Total confirmed cases across period: {analysis_df['confirmed_cases'].sum():,.0f}")
print(f"Highest month: {analysis_df.loc[analysis_df['confirmed_cases'].idxmax(), 'year_month']} "
      f"({analysis_df['confirmed_cases'].max():.0f} cases)")
print(f"Lowest month: {analysis_df.loc[analysis_df['confirmed_cases'].idxmin(), 'year_month']} "
      f"({analysis_df['confirmed_cases'].min():.0f} cases)")


In [ ]:
# ---------------------------------------------------------------------------
# 2. Yearly totals and year-over-year change
# ---------------------------------------------------------------------------

print()
print("=" * 70)
print("2. YEARLY TOTALS")
print("=" * 70)
yearly = analysis_df.groupby("year")["confirmed_cases"].agg(["sum", "mean", "max"]).round(1)
yearly.columns = ["total_confirmed", "monthly_avg", "peak_month_value"]
yearly["yoy_change_pct"] = yearly["total_confirmed"].pct_change().mul(100).round(1)
print(yearly.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 3. Seasonality: average confirmed cases by calendar month, across years
# ---------------------------------------------------------------------------

print()
print("=" * 70)
print("3. SEASONAL PATTERN (average confirmed cases by calendar month)")
print("=" * 70)
seasonal = analysis_df.groupby("month")["confirmed_cases"].agg(["mean", "std", "min", "max"]).round(1)
seasonal.index = [pd.Timestamp(2020, m, 1).strftime("%B") for m in seasonal.index]
print(seasonal.to_string())

peak_month = seasonal["mean"].idxmax()
trough_month = seasonal["mean"].idxmin()
print()
print(f"Peak season (avg): {peak_month} ({seasonal.loc[peak_month, 'mean']:.1f} cases/month avg)")
print(f"Trough season (avg): {trough_month} ({seasonal.loc[trough_month, 'mean']:.1f} cases/month avg)")


In [ ]:
# ---------------------------------------------------------------------------
# 4. Case Fatality Rate trend (from weekly data, aggregated to yearly)
# ---------------------------------------------------------------------------

weekly_df["year"] = weekly_df["report_date_iso"].dt.year
cfr_by_year = weekly_df[(weekly_df["year"] >= 2020) & (weekly_df["year"] <= 2025)].groupby("year").agg(
    total_confirmed=("confirmed_cases", "sum"),
    total_deaths=("deaths", "sum"),
)
cfr_by_year["cfr_percent"] = (cfr_by_year["total_deaths"] / cfr_by_year["total_confirmed"] * 100).round(1)

print()
print("=" * 70)
print("4. CASE FATALITY RATE BY YEAR (from weekly-level data)")
print("=" * 70)
print(cfr_by_year.to_string())


In [ ]:
# ---------------------------------------------------------------------------
# 5. Visualizations
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Panel 1: Full monthly series
axes[0, 0].plot(analysis_df["date"], analysis_df["confirmed_cases"], marker="o", markersize=3,
                 linewidth=1.2, color="#c0392b")
axes[0, 0].set_title("Monthly Confirmed Cases, 2020-2025")
axes[0, 0].set_ylabel("Confirmed cases")
axes[0, 0].grid(alpha=0.3)
axes[0, 0].tick_params(axis="x", rotation=45)
# Panel 2: Yearly totals bar chart
axes[0, 1].bar(yearly.index.astype(str), yearly["total_confirmed"], color="#2980b9")
axes[0, 1].set_title("Total Confirmed Cases by Year")
axes[0, 1].set_ylabel("Total confirmed cases")
for i, v in enumerate(yearly["total_confirmed"]):
    axes[0, 1].text(i, v + 5, f"{v:.0f}", ha="center", fontsize=9)
# Panel 3: Seasonal boxplot (distribution by calendar month across years)
month_order = [pd.Timestamp(2020, m, 1).strftime("%b") for m in range(1, 13)]
box_data = [analysis_df[analysis_df["month"] == m]["confirmed_cases"].values for m in range(1, 13)]
axes[1, 0].boxplot(box_data, tick_labels=month_order)
axes[1, 0].set_title("Seasonal Distribution by Calendar Month (all years)")
axes[1, 0].set_ylabel("Confirmed cases")
axes[1, 0].tick_params(axis="x", rotation=45)
# Panel 4: CFR trend by year
axes[1, 1].plot(cfr_by_year.index, cfr_by_year["cfr_percent"], marker="o", color="#8e44ad", linewidth=2)
axes[1, 1].set_title("Case Fatality Rate by Year")
axes[1, 1].set_ylabel("CFR (%)")
axes[1, 1].set_xlabel("Year")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
descriptive_plot_path = OUTPUTS_DIR / "descriptive_analysis_overview.png"
plt.savefig(descriptive_plot_path, dpi=120)
plt.show()

print()
print("Descriptive analysis plots saved to:", descriptive_plot_path.resolve())


In [ ]:
# Phase 6, Step 1: load monthly data, split into train (2020-2024) and
# hold-out (2025), and inspect the series before modeling.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"

df = pd.read_csv(monthly_csv_path)
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

print("Monthly dataset shape:", df.shape)
print()
print(df[["year_month", "confirmed_cases", "week_count", "status"]].to_string(index=False))
print()

if df["confirmed_cases"].isna().any():
    raise RuntimeError(
        "Missing confirmed_cases values found in the monthly dataset. "
        "Cannot proceed to modeling until every month has a value."
    )

train_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2024-12-31")].copy()
holdout_df = df[(df["date"] >= "2025-01-01") & (df["date"] <= "2025-12-31")].copy()
full_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-31")].copy()

print(f"Training set (2020-2024): {len(train_df)} months")
print(f"Hold-out set (2025): {len(holdout_df)} months")
print(f"Full 2020-2025 set: {len(full_df)} months")

if len(train_df) != 60:
    raise RuntimeError(f"Expected 60 training months, found {len(train_df)}. Check data before proceeding.")
if len(holdout_df) != 12:
    raise RuntimeError(f"Expected 12 hold-out months, found {len(holdout_df)}. Check data before proceeding.")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(full_df["date"], full_df["confirmed_cases"], marker="o", linewidth=1.5, color="#c0392b")
ax.axvline(pd.Timestamp("2025-01-01"), color="gray", linestyle="--", alpha=0.7, label="Train/Hold-out split")
ax.set_title("Monthly Confirmed Lassa Fever Cases, Nigeria (Jan 2020 - Dec 2025)")
ax.set_xlabel("Month")
ax.set_ylabel("Confirmed cases")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

plot_path = OUTPUTS_DIR / "monthly_series_overview.png"
plt.savefig(plot_path, dpi=120)
plt.show()

print()
print("Plot saved to:", plot_path.resolve())

In [ ]:
# Phase 6, Step 2: fit seasonal naive, SARIMA, and count-regression models
# on 2020-2024, evaluate all three against the 2025 hold-out.
# This cell is self-contained and imports everything it uses.

import warnings
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
df = pd.read_csv(monthly_csv_path)
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

train_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2024-12-31")].copy()
holdout_df = df[(df["date"] >= "2025-01-01") & (df["date"] <= "2025-12-31")].copy()

train_series = train_df.set_index("date")["confirmed_cases"]
holdout_series = holdout_df.set_index("date")["confirmed_cases"]
holdout_actual = holdout_series.values

# ---------------------------------------------------------------------------
# Model 1: Seasonal naive - forecast month m of 2025 = same calendar month
# of 2024 (the most recent full year available in training).
# ---------------------------------------------------------------------------

seasonal_naive_forecast = train_series[-12:].values  # Jan-Dec 2024, in order

# ---------------------------------------------------------------------------
# Model 2: SARIMA - parsimonious order given only 60 training months.
# (1,1,1)x(1,1,1,12) is a standard, modest starting specification.
# ---------------------------------------------------------------------------

sarima_model = SARIMAX(
    train_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)
sarima_forecast_result = sarima_fit.get_forecast(steps=12)
sarima_forecast = sarima_forecast_result.predicted_mean.values
sarima_forecast = np.clip(sarima_forecast, a_min=0, a_max=None)  # cases can't be negative

# ---------------------------------------------------------------------------
# Model 3: Count-data regression (Negative Binomial GLM) with a linear
# time trend and calendar-month seasonality (dummy variables).
# Negative Binomial chosen over Poisson since monthly case counts here
# are strongly overdispersed (variance >> mean, per the Jan/Feb spikes).
# ---------------------------------------------------------------------------

reg_df = train_df.copy()
reg_df["t"] = np.arange(len(reg_df))
reg_df["month"] = reg_df["date"].dt.month
month_dummies = pd.get_dummies(reg_df["month"], prefix="month", drop_first=True)
X_train = pd.concat([reg_df[["t"]], month_dummies], axis=1).astype(float)
X_train = sm.add_constant(X_train)
y_train = reg_df["confirmed_cases"].astype(float)

nb_model = sm.GLM(y_train, X_train, family=sm.families.NegativeBinomial())
nb_fit = nb_model.fit()

holdout_reg = holdout_df.copy()
holdout_reg["t"] = np.arange(len(reg_df), len(reg_df) + len(holdout_reg))
holdout_reg["month"] = holdout_reg["date"].dt.month
holdout_month_dummies = pd.get_dummies(holdout_reg["month"], prefix="month", drop_first=True)
# Ensure holdout has the same dummy columns as training (in case a month is missing)
for col in month_dummies.columns:
    if col not in holdout_month_dummies.columns:
        holdout_month_dummies[col] = 0
holdout_month_dummies = holdout_month_dummies[month_dummies.columns]

X_holdout = pd.concat([holdout_reg[["t"]], holdout_month_dummies], axis=1).astype(float)
X_holdout = sm.add_constant(X_holdout, has_constant="add")
X_holdout = X_holdout[X_train.columns]  # enforce identical column order

countreg_forecast = nb_fit.predict(X_holdout).values

# ---------------------------------------------------------------------------
# Evaluation metrics: MAE, RMSE, MAPE, MASE
# ---------------------------------------------------------------------------

def mae(actual, pred):
    return np.mean(np.abs(actual - pred))

def rmse(actual, pred):
    return np.sqrt(np.mean((actual - pred) ** 2))

def mape(actual, pred):
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100

def mase(actual, pred, train_values, seasonal_period=12):
    # Denominator: mean absolute error of the naive seasonal forecast
    # within the training set itself.
    naive_errors = np.abs(train_values[seasonal_period:] - train_values[:-seasonal_period])
    scale = np.mean(naive_errors)
    if scale == 0:
        return np.nan
    return np.mean(np.abs(actual - pred)) / scale

train_values = train_series.values

models = {
    "Seasonal Naive": seasonal_naive_forecast,
    "SARIMA(1,1,1)(1,1,1,12)": sarima_forecast,
    "Negative Binomial Regression": countreg_forecast,
}

results = []
for name, forecast in models.items():
    results.append({
        "model": name,
        "MAE": round(mae(holdout_actual, forecast), 2),
        "RMSE": round(rmse(holdout_actual, forecast), 2),
        "MAPE_%": round(mape(holdout_actual, forecast), 1),
        "MASE": round(mase(holdout_actual, forecast, train_values), 3),
    })

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)

print("=" * 70)
print("MODEL COMPARISON ON 2025 HOLD-OUT")
print("=" * 70)
print(results_df.to_string(index=False))
print()

best_model_name = results_df.iloc[0]["model"]
print(f"Selected model (lowest MAE on 2025 hold-out): {best_model_name}")

print()
print("Month-by-month comparison:")
comparison_df = pd.DataFrame({
    "month": holdout_df["year_month"].values,
    "actual": holdout_actual,
    "seasonal_naive": seasonal_naive_forecast.round(1),
    "sarima": sarima_forecast.round(1),
    "neg_binomial_reg": countreg_forecast.round(1),
})
print(comparison_df.to_string(index=False))

# Save comparison for the record
comparison_csv_path = OUTPUTS_DIR / "model_comparison_2025_holdout.csv"
comparison_df.to_csv(comparison_csv_path, index=False)
results_df.to_csv(OUTPUTS_DIR / "model_selection_metrics.csv", index=False)
print()
print("Saved:", comparison_csv_path.resolve())

In [ ]:
# Phase 6, Step 3: refit the selected model (Seasonal Naive) on the full
# 2020-2025 data, forecast Jan 2026-Dec 2028 with 95% prediction intervals,
# and save the locked forecast file.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
df = pd.read_csv(monthly_csv_path)
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

full_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-31")].copy()
full_series = full_df.set_index("date")["confirmed_cases"]

if len(full_series) != 72:
    raise RuntimeError(f"Expected 72 months for the full training period, found {len(full_series)}.")

# ---------------------------------------------------------------------------
# Seasonal naive forecast, 2026-2028: repeat the last fully observed
# seasonal cycle (2025's 12 months) for every future year. This is the
# standard extension of seasonal naive beyond a single forecast cycle -
# it does not cascade forecasts on top of forecasts, since we have no
# actual data to naive-copy from for 2026 or 2027 within the horizon.
# ---------------------------------------------------------------------------

last_year_values = full_series[-12:].values  # Jan-Dec 2025, in calendar order
forecast_months = pd.date_range("2026-01-01", "2028-12-01", freq="MS")

if len(forecast_months) != 36:
    raise RuntimeError(f"Expected 36 forecast months, generated {len(forecast_months)}.")

point_forecast = np.tile(last_year_values, 3)  # repeats 2025's pattern for 2026, 2027, 2028

# ---------------------------------------------------------------------------
# 95% prediction intervals: derived from in-sample seasonal naive residuals
# (actual[t] - actual[t-12]) across the full training period, following
# standard practice for naive-method interval estimation. Uncertainty
# widens for years further into the future (2027, 2028) since we are
# repeating the same base year with less justification the further out
# we go, reflected via a multiplier on the horizon in years.
# ---------------------------------------------------------------------------

seasonal_residuals = full_series.values[12:] - full_series.values[:-12]
residual_std = np.std(seasonal_residuals, ddof=1)

z_95 = stats.norm.ppf(0.975)

years_ahead = np.repeat([1, 2, 3], 12)  # 2026=1 year ahead of last obs cycle, 2027=2, 2028=3
interval_width = z_95 * residual_std * np.sqrt(years_ahead)

lower_bound = np.clip(point_forecast - interval_width, a_min=0, a_max=None)
upper_bound = point_forecast + interval_width

forecast_df = pd.DataFrame({
    "year_month": forecast_months.strftime("%Y-%m"),
    "point_forecast": np.round(point_forecast, 1),
    "lower_95": np.round(lower_bound, 1),
    "upper_95": np.round(upper_bound, 1),
    "model": "Seasonal Naive (selected: lowest MAE on 2025 hold-out)",
    "base_year_repeated": "2025",
})

print("=" * 70)
print("LOCKED FORECAST: January 2026 - December 2028")
print("=" * 70)
print(forecast_df.to_string(index=False))

# ---------------------------------------------------------------------------
# Save as the locked forecast file. Per spec, this file must never be
# overwritten once observed 2026+ data becomes available.
# ---------------------------------------------------------------------------

locked_forecast_path = OUTPUTS_DIR / "forecast_2026_2028_locked.csv"

if locked_forecast_path.exists():
    print()
    print(f"WARNING: {locked_forecast_path.name} already exists and will NOT be overwritten, "
          "per the locking requirement. Delete it manually first if you intend to regenerate it.")
else:
    forecast_df.to_csv(locked_forecast_path, index=False)
    print()
    print("Locked forecast saved to:", locked_forecast_path.resolve())

# Also save the model selection justification alongside it, for the audit trail
model_notes = {
    "selected_model": "Seasonal Naive",
    "selection_rule": "Lowest MAE on 2025 hold-out, pre-specified",
    "candidates_evaluated": "Seasonal Naive, SARIMA(1,1,1)(1,1,1,12), Negative Binomial regression (trend + month dummies)",
    "seasonal_naive_holdout_MAE": 36.67,
    "sarima_holdout_MAE": 80.93,
    "negbinom_holdout_MAE": 39.18,
    "training_period": "2020-01 to 2025-12 (72 months)",
    "forecast_horizon": "2026-01 to 2028-12 (36 months)",
    "note": (
        "SARIMA and the count-regression model were both outperformed by the "
        "simple seasonal naive benchmark on the 2025 hold-out, consistent "
        "with the small training-sample size (60-72 months) specified as a "
        "risk in the original project methodology. Seasonal naive selected "
        "per the pre-specified lowest-MAE rule."
    ),
}
notes_path = OUTPUTS_DIR / "model_selection_notes.csv"
pd.DataFrame([model_notes]).to_csv(notes_path, index=False)
print("Model selection notes saved to:", notes_path.resolve())

In [ ]:
# Phase 7, Step 2: aggregate the locked forecast to yearly totals (2026,
# 2027, 2028), compare observed 2026 data against the forecast month-by-
# month, compute metrics, and produce an explanatory chart.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

locked_forecast_path = OUTPUTS_DIR / "forecast_2026_2028_locked.csv"
observed_2026_path = PROCESSED_DIR / "monthly_2026_observed.csv"

forecast_df = pd.read_csv(locked_forecast_path)
forecast_df["date"] = pd.to_datetime(forecast_df["year_month"], format="%Y-%m")
forecast_df["year"] = forecast_df["date"].dt.year

observed_df = pd.read_csv(observed_2026_path)
observed_df["date"] = pd.to_datetime(observed_df["year_month"], format="%Y-%m")

# ---------------------------------------------------------------------------
# 1. Yearly forecast totals (2026, 2027, 2028)
# ---------------------------------------------------------------------------

yearly_forecast = forecast_df.groupby("year").agg(
    total_point_forecast=("point_forecast", "sum"),
    total_lower_95=("lower_95", "sum"),
    total_upper_95=("upper_95", "sum"),
).round(0)

print("=" * 70)
print("FORECAST TOTALS BY YEAR (2026-2028)")
print("=" * 70)
print(yearly_forecast.to_string())
print()

yearly_forecast_path = OUTPUTS_DIR / "forecast_yearly_totals.csv"
yearly_forecast.to_csv(yearly_forecast_path)
print("Saved to:", yearly_forecast_path.resolve())




In [ ]:
# Phase 7, Step 1: extract 2026 weekly confirmed-case data using the exact
# same validated dual-template logic from Phase 4, then aggregate to
# monthly totals for comparison against the locked forecast.
# This cell is self-contained and imports everything it uses.

import re
import csv
import logging
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

import pymupdf

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOGS_DIR = PROJECT_ROOT / "logs"

logger = logging.getLogger("lassa_fever_nimr")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOGS_DIR / "pipeline.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(ch)

catalogue_csv_path = INTERIM_DIR / "ncdc_catalogue.csv"
acquisition_log_path = INTERIM_DIR / "acquisition_log.csv"
corrections_csv_path = INTERIM_DIR / "manual_catalogue_corrections.csv"

with catalogue_csv_path.open("r", encoding="utf-8") as f:
    catalogue_rows = list(csv.DictReader(f))

with acquisition_log_path.open("r", encoding="utf-8") as f:
    log_rows = {row["report_id"]: row for row in csv.DictReader(f)}

with corrections_csv_path.open("r", encoding="utf-8") as f:
    corrections = {row["report_id"]: row for row in csv.DictReader(f)}

EXCLUDED_REPORT_IDS = {
    r["report_id"] for r in corrections.values()
    if r["correction_type"] in ("exclude_wrong_disease", "source_file_unrecoverable")
}

# 2026 reports only - not part of the training window, never processed before
reports_2026 = [
    r for r in catalogue_rows
    if log_rows.get(r["report_id"], {}).get("download_status") == "success"
    and r["report_date_iso"]
    and r["report_date_iso"][:4] == "2026"
    and r["report_id"] not in EXCLUDED_REPORT_IDS
]
reports_2026.sort(key=lambda r: r["report_date_iso"])

logger.info("Running extraction on %d reports from 2026.", len(reports_2026))

# ---------------------------------------------------------------------------
# Same validated extraction logic as Phase 4 (Step 4e / final version)
# ---------------------------------------------------------------------------

STANDARD_LABEL = re.compile(
    r"current\s*week\s*\(\s*(?:week\s*)?(\d{1,2})\s*\)\s*(?:indicators)?",
    re.IGNORECASE
)
EARLY_LABEL = re.compile(
    r"Summary\s*of\s*current\s*week\s*indicators", re.IGNORECASE
)
EPI_WEEK_HEADER = re.compile(r"Epi\s*Week\s*0?(\d{1,2})[:\s]", re.IGNORECASE)
NUMBER_TOKEN_PATTERN = re.compile(r"\d[\d,]*\.?\d*%?")

def extract_text(pdf_path, max_chars=8000):
    doc = pymupdf.open(pdf_path)
    text_parts, total_len = [], 0
    for page in doc:
        page_text = page.get_text()
        text_parts.append(page_text)
        total_len += len(page_text)
        if total_len >= max_chars:
            break
    doc.close()
    return "\n".join(text_parts)[:max_chars]

def clean_int(tok):
    cleaned = tok.replace(",", "").replace("%", "")
    if not cleaned:
        raise ValueError(f"Empty numeric token: {tok!r}")
    return int(float(cleaned))

def extract_weekly_confirmed(text):
    for match in STANDARD_LABEL.finditer(text):
        tail = text[match.end():match.end() + 300]
        token_matches = list(NUMBER_TOKEN_PATTERN.finditer(tail))
        if len(token_matches) < 5:
            continue
        span_to_fifth_token = tail[:token_matches[4].end()]
        if "cumulative" in span_to_fifth_token.lower():
            continue
        tokens = [m.group(0) for m in token_matches[:5]]
        return {
            "extraction_format": "standard_5field",
            "matched_week": match.group(1),
            "suspected_cases": clean_int(tokens[0]),
            "confirmed_cases": clean_int(tokens[1]),
            "probable_cases": clean_int(tokens[2]),
            "deaths": clean_int(tokens[3]),
            "cfr_percent": tokens[4].replace("%", ""),
        }

    early_match = EARLY_LABEL.search(text)
    if early_match:
        tail = text[early_match.end():early_match.end() + 250]
        tokens = NUMBER_TOKEN_PATTERN.findall(tail)
        if len(tokens) >= 4:
            week_match = EPI_WEEK_HEADER.search(text)
            matched_week = week_match.group(1) if week_match else None
            return {
                "extraction_format": "early2020_4field_no_probable",
                "matched_week": matched_week,
                "suspected_cases": clean_int(tokens[0]),
                "confirmed_cases": clean_int(tokens[2]),
                "probable_cases": 0,
                "deaths": clean_int(tokens[3]),
                "cfr_percent": tokens[4].replace("%", "") if len(tokens) > 4 else None,
            }

    return None

results_2026 = []
failures_2026 = []
flags_2026 = []

for row in reports_2026:
    report_id = row["report_id"]
    local_path = Path(log_rows[report_id]["local_file"])
    text = extract_text(local_path)

    try:
        result = extract_weekly_confirmed(text)
    except ValueError as exc:
        failures_2026.append({
            "report_id": report_id, "report_title": row["report_title"],
            "report_date_iso": row["report_date_iso"], "reason": f"error: {exc}",
        })
        continue

    if result is None:
        failures_2026.append({
            "report_id": report_id, "report_title": row["report_title"],
            "report_date_iso": row["report_date_iso"], "reason": "no_pattern_matched",
        })
        continue

    week_matches = (result["matched_week"] or "").lstrip("0") == row["sitrep_week"].lstrip("0")

    entry = {
        "report_id": report_id,
        "report_title": row["report_title"],
        "report_date_iso": row["report_date_iso"],
        "sitrep_week": row["sitrep_week"],
        "extraction_format": result["extraction_format"],
        "confirmed_cases": result["confirmed_cases"],
        "suspected_cases": result["suspected_cases"],
        "deaths": result["deaths"],
        "week_matches_catalogue": week_matches,
    }
    results_2026.append(entry)

    flags = []
    if not week_matches:
        flags.append("week_mismatch")
    if result["confirmed_cases"] == 0:
        flags.append("zero_confirmed")
    if result["confirmed_cases"] > 300:
        flags.append("implausibly_large_confirmed")
    if flags:
        flags_2026.append({**entry, "flags": ";".join(flags)})

print("=" * 70)
print("2026 EXTRACTION SUMMARY")
print("=" * 70)
print(f"Total 2026 reports processed: {len(reports_2026)}")
print(f"Successfully extracted: {len(results_2026)}")
print(f"Failed: {len(failures_2026)}")
print(f"Flagged for review: {len(flags_2026)}")
print()

if failures_2026:
    print("Failed reports:")
    for f in failures_2026:
        print(f"  - {f['report_title']} | {f['report_date_iso']} | {f['reason']}")
    print()

if flags_2026:
    print("Flagged reports (excluded from monthly totals, same as training-window policy):")
    for f in flags_2026:
        print(f"  - {f['report_title']} | {f['report_date_iso']} | flags: {f['flags']} | confirmed={f['confirmed_cases']}")
    print()

# ---------------------------------------------------------------------------
# Save 2026 weekly extraction, then aggregate approved weeks to monthly
# totals (same policy as training: exclude flagged/failed).
# ---------------------------------------------------------------------------

weekly_2026_path = INTERIM_DIR / "weekly_2026_extracted.csv"
if results_2026:
    with weekly_2026_path.open("w", newline="", encoding="utf-8") as f:
        fieldnames = list(results_2026[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results_2026)

flagged_ids_2026 = {f["report_id"] for f in flags_2026}
approved_2026 = [r for r in results_2026 if r["report_id"] not in flagged_ids_2026]

monthly_2026 = defaultdict(lambda: {"confirmed_cases": 0, "week_count": 0})
for row in approved_2026:
    month_key = row["report_date_iso"][:7]
    monthly_2026[month_key]["confirmed_cases"] += row["confirmed_cases"]
    monthly_2026[month_key]["week_count"] += 1

monthly_2026_rows = [
    {"year_month": k, "confirmed_cases": v["confirmed_cases"], "week_count": v["week_count"]}
    for k, v in sorted(monthly_2026.items())
]

observed_2026_path = PROCESSED_DIR / "monthly_2026_observed.csv"
with observed_2026_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(monthly_2026_rows[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(monthly_2026_rows)

print("Monthly 2026 observed data:")
for r in monthly_2026_rows:
    print(f"  {r['year_month']}: confirmed={r['confirmed_cases']}, weeks={r['week_count']}")
print()
print("Observed 2026 monthly data saved to:", observed_2026_path.resolve())

In [ ]:
# Phase 6, Step 3c: refit the trend-aware Negative Binomial regression on
# the full 2020-2025 data, forecast 2026-2028, and save as a SEPARATE
# supplementary file alongside the locked Seasonal Naive forecast.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
df = pd.read_csv(monthly_csv_path)
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

full_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-31")].copy()
if len(full_df) != 72:
    raise RuntimeError(f"Expected 72 months, found {len(full_df)}.")

# ---------------------------------------------------------------------------
# Refit on the FULL 2020-2025 period (same specification validated earlier)
# ---------------------------------------------------------------------------

full_df["t"] = np.arange(len(full_df))
full_df["month"] = full_df["date"].dt.month
month_dummies = pd.get_dummies(full_df["month"], prefix="month", drop_first=True)
X_full = pd.concat([full_df[["t"]], month_dummies], axis=1).astype(float)
X_full = sm.add_constant(X_full)
y_full = full_df["confirmed_cases"].astype(float)

nb_model = sm.GLM(y_full, X_full, family=sm.families.NegativeBinomial())
nb_fit = nb_model.fit()

# ---------------------------------------------------------------------------
# Forecast 2026-2028
# ---------------------------------------------------------------------------

forecast_months = pd.date_range("2026-01-01", "2028-12-01", freq="MS")
future_df = pd.DataFrame({"date": forecast_months})
future_df["t"] = np.arange(len(full_df), len(full_df) + len(future_df))
future_df["month"] = future_df["date"].dt.month

future_month_dummies = pd.get_dummies(future_df["month"], prefix="month", drop_first=True)
for col in month_dummies.columns:
    if col not in future_month_dummies.columns:
        future_month_dummies[col] = 0
future_month_dummies = future_month_dummies[month_dummies.columns]

X_future = pd.concat([future_df[["t"]], future_month_dummies], axis=1).astype(float)
X_future = sm.add_constant(X_future, has_constant="add")
X_future = X_future[X_full.columns]

pred_result = nb_fit.get_prediction(X_future)
nb_point = pred_result.predicted_mean
nb_ci = pred_result.conf_int(alpha=0.05)  # 95% CI

nb_forecast_df = pd.DataFrame({
    "year_month": forecast_months.strftime("%Y-%m"),
    "point_forecast": np.round(nb_point, 1),
    "lower_95": np.round(np.clip(nb_ci[:, 0], 0, None), 1),
    "upper_95": np.round(nb_ci[:, 1], 1),
    "model": "Negative Binomial Regression (trend + seasonality) - SUPPLEMENTARY, not the locked forecast",
})

nb_forecast_df["year"] = pd.to_datetime(nb_forecast_df["year_month"], format="%Y-%m").dt.year
nb_yearly = nb_forecast_df.groupby("year").agg(
    total_point_forecast=("point_forecast", "sum"),
    total_lower_95=("lower_95", "sum"),
    total_upper_95=("upper_95", "sum"),
).round(0)

print("=" * 70)
print("SUPPLEMENTARY FORECAST: Negative Binomial (trend-aware) - YEARLY TOTALS")
print("=" * 70)
print(nb_yearly.to_string())
print()
print("Compare to locked Seasonal Naive: 1,131 in every year (flat, no trend)")

supplementary_path = OUTPUTS_DIR / "forecast_2026_2028_supplementary_trend_model.csv"
nb_forecast_df.drop(columns=["year"]).to_csv(supplementary_path, index=False)
print()
print("Supplementary forecast saved to:", supplementary_path.resolve())
print("(This does NOT replace the locked forecast_2026_2028_locked.csv)")

In [ ]:
# Additional candidate models: Polynomial (quadratic) trend regression and
# Holt-Winters Exponential Smoothing, both using the FULL 2020-2025 history
# (not just 2025). Evaluated on the 2025 hold-out the same way as before,
# for a fair comparison against Seasonal Naive and the linear-trend model.
# This cell is self-contained and imports everything it uses.

import warnings
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
df = pd.read_csv(monthly_csv_path)
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

train_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2024-12-31")].copy()
holdout_df = df[(df["date"] >= "2025-01-01") & (df["date"] <= "2025-12-31")].copy()
holdout_actual = holdout_df["confirmed_cases"].values

# ---------------------------------------------------------------------------
# Model A: Polynomial (quadratic) trend + calendar-month seasonality,
# Negative Binomial GLM. Uses ALL 60 training months (2020-2024), not just
# the most recent year. The t^2 term lets growth accelerate or decelerate,
# rather than assuming a constant straight-line rate like the linear model.
# ---------------------------------------------------------------------------

train_df["t"] = np.arange(len(train_df))
train_df["t_squared"] = train_df["t"] ** 2
train_df["month"] = train_df["date"].dt.month
month_dummies = pd.get_dummies(train_df["month"], prefix="month", drop_first=True)

X_train_poly = pd.concat([train_df[["t", "t_squared"]], month_dummies], axis=1).astype(float)
X_train_poly = sm.add_constant(X_train_poly)
y_train = train_df["confirmed_cases"].astype(float)

poly_model = sm.GLM(y_train, X_train_poly, family=sm.families.NegativeBinomial())
poly_fit = poly_model.fit()

holdout_df["t"] = np.arange(len(train_df), len(train_df) + len(holdout_df))
holdout_df["t_squared"] = holdout_df["t"] ** 2
holdout_df["month"] = holdout_df["date"].dt.month
holdout_month_dummies = pd.get_dummies(holdout_df["month"], prefix="month", drop_first=True)
for col in month_dummies.columns:
    if col not in holdout_month_dummies.columns:
        holdout_month_dummies[col] = 0
holdout_month_dummies = holdout_month_dummies[month_dummies.columns]

X_holdout_poly = pd.concat([holdout_df[["t", "t_squared"]], holdout_month_dummies], axis=1).astype(float)
X_holdout_poly = sm.add_constant(X_holdout_poly, has_constant="add")
X_holdout_poly = X_holdout_poly[X_train_poly.columns]

poly_forecast = poly_fit.predict(X_holdout_poly).values

# ---------------------------------------------------------------------------
# Model B: Holt-Winters Exponential Smoothing (additive trend, additive
# seasonality, 12-month period). Also uses all 60 training months - it
# builds its trend and seasonal estimates from the entire series, updating
# them progressively month by month through the whole training period.
# ---------------------------------------------------------------------------

train_series = train_df.set_index("date")["confirmed_cases"]

hw_model = ExponentialSmoothing(
    train_series,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
)
hw_fit = hw_model.fit()
hw_forecast = hw_fit.forecast(12).values
hw_forecast = np.clip(hw_forecast, a_min=0, a_max=None)

# ---------------------------------------------------------------------------
# Evaluate both against the 2025 hold-out, same metrics as before
# ---------------------------------------------------------------------------

def mae(actual, pred):
    return np.mean(np.abs(actual - pred))

def rmse(actual, pred):
    return np.sqrt(np.mean((actual - pred) ** 2))

def mape(actual, pred):
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100

new_models = {
    "Polynomial (quadratic) trend + seasonality": poly_forecast,
    "Holt-Winters Exponential Smoothing": hw_forecast,
}

print("=" * 70)
print("NEW CANDIDATE MODELS vs 2025 HOLD-OUT")
print("=" * 70)
results = []
for name, forecast in new_models.items():
    results.append({
        "model": name,
        "MAE": round(mae(holdout_actual, forecast), 2),
        "RMSE": round(rmse(holdout_actual, forecast), 2),
        "MAPE_%": round(mape(holdout_actual, forecast), 1),
    })

new_results_df = pd.DataFrame(results)
print(new_results_df.to_string(index=False))
print()
print("For reference, previously tested models on the same 2025 hold-out:")
print("  Seasonal Naive:                MAE=36.67")
print("  SARIMA(1,1,1)(1,1,1,12):       MAE=80.93")
print("  Negative Binomial (linear t):  MAE=39.18")
print()

print("Month-by-month comparison:")
comparison_df = pd.DataFrame({
    "month": holdout_df["year_month"].values,
    "actual": holdout_actual,
    "polynomial_trend": poly_forecast.round(1),
    "holt_winters": hw_forecast.round(1),
})
print(comparison_df.to_string(index=False))

new_comparison_path = OUTPUTS_DIR / "additional_models_2025_holdout.csv"
comparison_df.to_csv(new_comparison_path, index=False)
print()
print("Saved to:", new_comparison_path.resolve())

In [ ]:
# Refit the Polynomial (quadratic) trend model on the FULL 2020-2025 data,
# then forecast 2026-2028. This just shows the prediction - no comparison
# against Seasonal Naive or anything else yet.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
df = pd.read_csv(monthly_csv_path)
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

full_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-31")].copy()
if len(full_df) != 72:
    raise RuntimeError(f"Expected 72 months, found {len(full_df)}.")

# ---------------------------------------------------------------------------
# Fit on ALL 72 months (2020-2025), quadratic time trend + month seasonality
# ---------------------------------------------------------------------------

full_df["t"] = np.arange(len(full_df))
full_df["t_squared"] = full_df["t"] ** 2
full_df["month"] = full_df["date"].dt.month
month_dummies = pd.get_dummies(full_df["month"], prefix="month", drop_first=True)

X_full = pd.concat([full_df[["t", "t_squared"]], month_dummies], axis=1).astype(float)
X_full = sm.add_constant(X_full)
y_full = full_df["confirmed_cases"].astype(float)

poly_model = sm.GLM(y_full, X_full, family=sm.families.NegativeBinomial())
poly_fit = poly_model.fit()

# ---------------------------------------------------------------------------
# Forecast 2026-2028 (36 months)
# ---------------------------------------------------------------------------

forecast_months = pd.date_range("2026-01-01", "2028-12-01", freq="MS")
future_df = pd.DataFrame({"date": forecast_months})
future_df["t"] = np.arange(len(full_df), len(full_df) + len(future_df))
future_df["t_squared"] = future_df["t"] ** 2
future_df["month"] = future_df["date"].dt.month
future_df["year"] = future_df["date"].dt.year

future_month_dummies = pd.get_dummies(future_df["month"], prefix="month", drop_first=True)
for col in month_dummies.columns:
    if col not in future_month_dummies.columns:
        future_month_dummies[col] = 0
future_month_dummies = future_month_dummies[month_dummies.columns]

X_future = pd.concat([future_df[["t", "t_squared"]], future_month_dummies], axis=1).astype(float)
X_future = sm.add_constant(X_future, has_constant="add")
X_future = X_future[X_full.columns]

pred_result = poly_fit.get_prediction(X_future)
future_df["point_forecast"] = pred_result.predicted_mean.round(1)
ci = pred_result.conf_int(alpha=0.05)
future_df["lower_95"] = np.clip(ci[:, 0], 0, None).round(1)
future_df["upper_95"] = ci[:, 1].round(1)

future_df["year_month"] = future_df["date"].dt.strftime("%Y-%m")

print("=" * 70)
print("POLYNOMIAL (QUADRATIC) TREND MODEL - FULL 2026-2028 FORECAST")
print("=" * 70)
print(future_df[["year_month", "point_forecast", "lower_95", "upper_95"]].to_string(index=False))
print()

# ---------------------------------------------------------------------------
# Yearly totals
# ---------------------------------------------------------------------------

yearly_totals = future_df.groupby("year").agg(
    total_point_forecast=("point_forecast", "sum"),
    total_lower_95=("lower_95", "sum"),
    total_upper_95=("upper_95", "sum"),
).round(0)

print("=" * 70)
print("YEARLY TOTALS (Polynomial trend model)")
print("=" * 70)
print(yearly_totals.to_string())

# Save it, clearly labeled as its own separate file
poly_forecast_path = OUTPUTS_DIR / "forecast_2026_2028_polynomial_trend_model.csv"
future_df[["year_month", "point_forecast", "lower_95", "upper_95"]].to_csv(poly_forecast_path, index=False)
print()
print("Saved to:", poly_forecast_path.resolve())

In [ ]:
# Compare all four candidate models' predictions for 2026 against the
# REAL observed 2026 data we already extracted (through August/Week 34).
# This is a direct test: did each model's forecast for 2026 come close
# to what actually happened in 2026, for the months we can check.
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
observed_2026_path = PROCESSED_DIR / "monthly_2026_observed.csv"

df = pd.read_csv(monthly_csv_path)
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

full_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-31")].copy()
full_df["t"] = np.arange(len(full_df))
full_df["month"] = full_df["date"].dt.month
month_dummies = pd.get_dummies(full_df["month"], prefix="month", drop_first=True)

observed_2026 = pd.read_csv(observed_2026_path)

forecast_months_2026 = pd.date_range("2026-01-01", "2026-12-01", freq="MS")

def build_future_X(months, t_start, month_dummy_cols, include_t_squared=False):
    fdf = pd.DataFrame({"date": months})
    fdf["t"] = np.arange(t_start, t_start + len(fdf))
    if include_t_squared:
        fdf["t_squared"] = fdf["t"] ** 2
    fdf["month"] = fdf["date"].dt.month
    dummies = pd.get_dummies(fdf["month"], prefix="month", drop_first=True)
    for col in month_dummy_cols:
        if col not in dummies.columns:
            dummies[col] = 0
    dummies = dummies[month_dummy_cols]
    cols = ["t", "t_squared"] if include_t_squared else ["t"]
    X = pd.concat([fdf[cols], dummies], axis=1).astype(float)
    return fdf, X

# --- Model 1: Seasonal Naive ---
last_year_values = full_df.set_index("date")["confirmed_cases"][-12:].values
seasonal_naive_2026 = last_year_values

# --- Model 2: Negative Binomial, linear trend ---
X_full_lin = sm.add_constant(pd.concat([full_df[["t"]], month_dummies], axis=1).astype(float))
y_full = full_df["confirmed_cases"].astype(float)
nb_lin_fit = sm.GLM(y_full, X_full_lin, family=sm.families.NegativeBinomial()).fit()
_, X_future_lin = build_future_X(forecast_months_2026, len(full_df), month_dummies.columns)
X_future_lin = sm.add_constant(X_future_lin, has_constant="add")[X_full_lin.columns]
nb_lin_2026 = nb_lin_fit.predict(X_future_lin).values

# --- Model 3: Negative Binomial, polynomial (quadratic) trend ---
full_df["t_squared"] = full_df["t"] ** 2
X_full_poly = sm.add_constant(pd.concat([full_df[["t", "t_squared"]], month_dummies], axis=1).astype(float))
nb_poly_fit = sm.GLM(y_full, X_full_poly, family=sm.families.NegativeBinomial()).fit()
_, X_future_poly = build_future_X(forecast_months_2026, len(full_df), month_dummies.columns, include_t_squared=True)
X_future_poly = sm.add_constant(X_future_poly, has_constant="add")[X_full_poly.columns]
nb_poly_2026 = nb_poly_fit.predict(X_future_poly).values

# --- Model 4: Holt-Winters ---
full_series = full_df.set_index("date")["confirmed_cases"]
hw_fit = ExponentialSmoothing(
    full_series, trend="add", seasonal="add", seasonal_periods=12,
    initialization_method="estimated",
).fit()
hw_2026 = np.clip(hw_fit.forecast(12).values, 0, None)

# ---------------------------------------------------------------------------
# Assemble full 2026 forecast table for all 4 models
# ---------------------------------------------------------------------------

all_forecasts_2026 = pd.DataFrame({
    "year_month": forecast_months_2026.strftime("%Y-%m"),
    "seasonal_naive": seasonal_naive_2026.round(1),
    "neg_binomial_linear": nb_lin_2026.round(1),
    "neg_binomial_polynomial": nb_poly_2026.round(1),
    "holt_winters": hw_2026.round(1),
})

# ---------------------------------------------------------------------------
# Merge with REAL observed 2026 data and compute error for each model,
# for only the months we actually have real data for.
# ---------------------------------------------------------------------------

comparison = observed_2026.merge(all_forecasts_2026, on="year_month", how="left")
comparison = comparison.dropna(subset=["seasonal_naive"]).reset_index(drop=True)

print("=" * 90)
print(f"ALL MODELS vs REAL 2026 DATA ({len(comparison)} months observed so far)")
print("=" * 90)
print(comparison[["year_month", "confirmed_cases", "seasonal_naive",
                   "neg_binomial_linear", "neg_binomial_polynomial", "holt_winters"]].to_string(index=False))
print()

print("=" * 90)
print("ERROR SUMMARY: which model's 2026 predictions were closest to reality?")
print("=" * 90)
actual = comparison["confirmed_cases"].values
for model_col in ["seasonal_naive", "neg_binomial_linear", "neg_binomial_polynomial", "holt_winters"]:
    pred = comparison[model_col].values
    mae_val = np.mean(np.abs(actual - pred))
    rmse_val = np.sqrt(np.mean((actual - pred) ** 2))
    print(f"  {model_col:30s}  MAE={mae_val:7.1f}   RMSE={rmse_val:7.1f}")

comparison_out_path = OUTPUTS_DIR / "all_models_vs_actual_2026.csv"
comparison.to_csv(comparison_out_path, index=False)
print()
print("Saved to:", comparison_out_path.resolve())

In [ ]:
# Compare all four candidate models' predictions for 2026 against the
# REAL observed 2026 data we already extracted (through August/Week 34).
# This cell is self-contained and imports everything it uses.

import csv
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

monthly_csv_path = PROCESSED_DIR / "monthly_confirmed_cases.csv"
observed_2026_path = PROCESSED_DIR / "monthly_2026_observed.csv"

df = pd.read_csv(monthly_csv_path)
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

full_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-31")].copy()
full_df["t"] = np.arange(len(full_df))
full_df["month"] = full_df["date"].dt.month
month_dummies = pd.get_dummies(full_df["month"], prefix="month", drop_first=True)

observed_2026 = pd.read_csv(observed_2026_path)

forecast_months_2026 = pd.date_range("2026-01-01", "2026-12-01", freq="MS")

def build_future_X(months, t_start, month_dummy_cols, include_t_squared=False):
    fdf = pd.DataFrame({"date": months})
    fdf["t"] = np.arange(t_start, t_start + len(fdf))
    if include_t_squared:
        fdf["t_squared"] = fdf["t"] ** 2
    fdf["month"] = fdf["date"].dt.month
    dummies = pd.get_dummies(fdf["month"], prefix="month", drop_first=True)
    for col in month_dummy_cols:
        if col not in dummies.columns:
            dummies[col] = 0
    dummies = dummies[month_dummy_cols]
    cols = ["t", "t_squared"] if include_t_squared else ["t"]
    X = pd.concat([fdf[cols], dummies], axis=1).astype(float)
    return fdf, X

# --- Model 1: Seasonal Naive ---
last_year_values = full_df.set_index("date")["confirmed_cases"][-12:].values
seasonal_naive_2026 = last_year_values

# --- Model 2: Negative Binomial, linear trend ---
X_full_lin = sm.add_constant(pd.concat([full_df[["t"]], month_dummies], axis=1).astype(float))
y_full = full_df["confirmed_cases"].astype(float)
nb_lin_fit = sm.GLM(y_full, X_full_lin, family=sm.families.NegativeBinomial()).fit()
_, X_future_lin = build_future_X(forecast_months_2026, len(full_df), month_dummies.columns)
X_future_lin = sm.add_constant(X_future_lin, has_constant="add")[X_full_lin.columns]
nb_lin_2026 = nb_lin_fit.predict(X_future_lin).values

# --- Model 3: Negative Binomial, polynomial (quadratic) trend ---
full_df["t_squared"] = full_df["t"] ** 2
X_full_poly = sm.add_constant(pd.concat([full_df[["t", "t_squared"]], month_dummies], axis=1).astype(float))
nb_poly_fit = sm.GLM(y_full, X_full_poly, family=sm.families.NegativeBinomial()).fit()
_, X_future_poly = build_future_X(forecast_months_2026, len(full_df), month_dummies.columns, include_t_squared=True)
X_future_poly = sm.add_constant(X_future_poly, has_constant="add")[X_full_poly.columns]
nb_poly_2026 = nb_poly_fit.predict(X_future_poly).values

# --- Model 4: Holt-Winters ---
full_series = full_df.set_index("date")["confirmed_cases"]
hw_fit = ExponentialSmoothing(
    full_series, trend="add", seasonal="add", seasonal_periods=12,
    initialization_method="estimated",
).fit()
hw_2026 = np.clip(hw_fit.forecast(12).values, 0, None)

# ---------------------------------------------------------------------------
# Assemble full 2026 forecast table for all 4 models
# ---------------------------------------------------------------------------

all_forecasts_2026 = pd.DataFrame({
    "year_month": forecast_months_2026.strftime("%Y-%m"),
    "seasonal_naive": seasonal_naive_2026.round(1),
    "neg_binomial_linear": nb_lin_2026.round(1),
    "neg_binomial_polynomial": nb_poly_2026.round(1),
    "holt_winters": hw_2026.round(1),
})

# ---------------------------------------------------------------------------
# Merge with REAL observed 2026 data and compute error for each model,
# for only the months we actually have real data for.
# ---------------------------------------------------------------------------

comparison = observed_2026.merge(all_forecasts_2026, on="year_month", how="left")
comparison = comparison.dropna(subset=["seasonal_naive"]).reset_index(drop=True)

print("=" * 90)
print(f"ALL MODELS vs REAL 2026 DATA ({len(comparison)} months observed so far)")
print("=" * 90)
print(comparison[["year_month", "confirmed_cases", "seasonal_naive",
                   "neg_binomial_linear", "neg_binomial_polynomial", "holt_winters"]].to_string(index=False))
print()

print("=" * 90)
print("Which model's 2026 predictions were closest to reality?")
print("=" * 90)
actual = comparison["confirmed_cases"].values
for model_col in ["seasonal_naive", "neg_binomial_linear", "neg_binomial_polynomial", "holt_winters"]:
    pred = comparison[model_col].values
    mae_val = np.mean(np.abs(actual - pred))
    rmse_val = np.sqrt(np.mean((actual - pred) ** 2))
    print(f"  {model_col:30s}  MAE={mae_val:7.1f}   RMSE={rmse_val:7.1f}")

comparison_out_path = OUTPUTS_DIR / "all_models_vs_actual_2026.csv"
comparison.to_csv(comparison_out_path, index=False)
print()
print("Saved to:", comparison_out_path.resolve())

In [ ]:
# Trace the 2026 gap (1,031 vs NCDC's 1,056) and draw the model comparison chart.
# This cell is self-contained and imports everything it uses.

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

weekly = pd.read_csv(INTERIM_DIR / "weekly_2026_extracted.csv")
weekly["sitrep_week"] = weekly["sitrep_week"].astype(int)
weekly = weekly.sort_values("sitrep_week")

expected_weeks = set(range(1, 35))
missing_weeks = sorted(expected_weeks - set(weekly["sitrep_week"]))
flagged = weekly[(weekly["confirmed_cases"] == 0) | (weekly["week_matches_catalogue"] == False)]

print("Weeks extracted:", len(weekly), "of 34 expected")
print("Missing weeks:", missing_weeks)
print("Sum of ALL extracted weekly confirmed cases:", weekly["confirmed_cases"].sum())
print("NCDC cumulative stated in the week 34 report: 1056")
print()
print("Flagged 2026 rows (excluded from the monthly totals):")
print(flagged[["report_title", "report_date_iso", "confirmed_cases", "week_matches_catalogue"]].to_string(index=False))

comp = pd.read_csv(OUTPUTS_DIR / "all_models_vs_actual_2026.csv")
comp["date"] = pd.to_datetime(comp["year_month"], format="%Y-%m")

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(comp["date"], comp["confirmed_cases"], color="black", linewidth=3, marker="o", label="Observed 2026")
ax.plot(comp["date"], comp["seasonal_naive"], linestyle="--", marker="o", label="Seasonal Naive")
ax.plot(comp["date"], comp["holt_winters"], linestyle="--", marker="o", label="Holt-Winters")
ax.plot(comp["date"], comp["neg_binomial_linear"], linestyle=":", marker="o", label="Neg. Binomial (linear trend)")
ax.plot(comp["date"], comp["neg_binomial_polynomial"], linestyle=":", marker="o", label="Neg. Binomial (quadratic trend)")
ax.set_title("Reported laboratory-confirmed Lassa fever cases, 2026: observed vs model predictions")
ax.set_xlabel("Month")
ax.set_ylabel("Confirmed cases")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
chart_path = OUTPUTS_DIR / "all_models_vs_actual_2026_chart.png"
plt.savefig(chart_path, dpi=120)
plt.show()
print("Chart saved to:", chart_path.resolve())

In [ ]:
# Holt-Winters forecast for 2026-2028, refit on the full 2020-2025 data.
# Saved as a separate supplementary file; does not touch the locked forecast.
# This cell is self-contained and imports everything it uses.

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd() / "lassa_fever_nimr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

df = pd.read_csv(PROCESSED_DIR / "monthly_confirmed_cases.csv")
df["confirmed_cases"] = pd.to_numeric(df["confirmed_cases"], errors="coerce")
df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

full_df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-31")].copy()
if len(full_df) != 72:
    raise RuntimeError(f"Expected 72 months, found {len(full_df)}.")

full_series = full_df.set_index("date")["confirmed_cases"].astype(float)
full_series.index.freq = "MS"

# Same specification used in the hold-out and 2026 comparison cells
hw_fit = ExponentialSmoothing(
    full_series,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
).fit()

forecast_months = pd.date_range("2026-01-01", "2028-12-01", freq="MS")
point = np.clip(hw_fit.forecast(36).values, 0, None)

# 95% intervals from 2000 simulated future paths (fixed seed for reproducibility)
sims = hw_fit.simulate(
    nsimulations=36, repetitions=2000, error="add",
    anchor="end", random_state=42,
)
sims = np.clip(np.asarray(sims), 0, None)
lower = np.percentile(sims, 2.5, axis=1)
upper = np.percentile(sims, 97.5, axis=1)

forecast_df = pd.DataFrame({
    "year_month": forecast_months.strftime("%Y-%m"),
    "point_forecast": point.round(1),
    "lower_95": lower.round(1),
    "upper_95": upper.round(1),
})
forecast_df["year"] = forecast_months.year

print("=" * 70)
print("HOLT-WINTERS FORECAST, 2026-2028")
print("=" * 70)
print(forecast_df[["year_month", "point_forecast", "lower_95", "upper_95"]].to_string(index=False))

# Yearly totals: interval taken from the simulated yearly totals, not by summing monthly bounds
years = forecast_months.year.values
rows = []
for yr in [2026, 2027, 2028]:
    mask = years == yr
    sim_totals = sims[mask, :].sum(axis=0)
    rows.append({
        "year": yr,
        "total_point_forecast": round(point[mask].sum()),
        "total_lower_95": round(np.percentile(sim_totals, 2.5)),
        "total_upper_95": round(np.percentile(sim_totals, 97.5)),
    })
yearly = pd.DataFrame(rows)

print()
print("=" * 70)
print("YEARLY TOTALS (Holt-Winters)")
print("=" * 70)
print(yearly.to_string(index=False))

out_path = OUTPUTS_DIR / "forecast_2026_2028_holt_winters_supplementary.csv"
forecast_df.drop(columns=["year"]).to_csv(out_path, index=False)
print()
print("Saved to:", out_path.resolve())